# Master Experiment Notebook
## Uncertainty-Aware, Network-Feasible, Decentralized EV Charging (journal resubmission)

This notebook runs **every experiment** in the revision plan, from raw data to statistics, figures,
red-team checks, and a reviewer-response matrix filled with your own numbers.

| Block | What it does | Answers |
|---|---|---|
| **E1** | Real ACN sessions on their real dates (2019), aligned with NSRDB 2019 and CAISO 2019. No resampling, no synthetic sessions. | R2-intro, R3.5 |
| **E2** | IEEE 33-bus feeder (pandapower), garage at a weak bus, LinDistFlow constraints for scheduling, **AC power flow** for evaluation, site transformer, branch ratings, stress tests. | R1.2, R2.1, R2.3, R3.5 |
| **E6** | Forecasting with a stated horizon (1 h and day-ahead), all inputs lagged at least the horizon (asserted in code), smart-persistence baselines, skill scores, daytime-only metrics, CQR intervals. | R1.3, R2.6 |
| **E3** | Proposed **decentralized protocol**: ADMM sharing (Boyd et al., 2011, §7.3). EV agents keep departure time and energy private; the coordinator broadcasts congestion prices. | R1.1, R3.1, R3.2 |
| **E4** | Baselines under *identical information and constraints*: Uncontrolled, EDF, LLF, centralized MPC (point forecast, ASA-style), centralized MPC (conformal), offline LP optimum (perfect foresight). Cost per kWh *delivered*. | R1.5, R2.4, R3.4 |
| **E5** | Closed loop: point vs raw quantile vs static / expanding / rolling conformal vs ACI. Measured on **operational** outcomes (AC violations, cost, energy). | R1.4, R2.2, R3.3 |
| **E7** | Day-level paired statistics (moving-block bootstrap CIs, Wilcoxon with Holm correction), both sites, honest disclosure of shared inputs. | R2.5, R2.3 |
| **W9** | Red-team checks (leakage, chronology, identical constraints, binding constraints, linearization error) and an auto-filled reviewer-response matrix. | all |

**How to run**
0. `pip install numpy pandas scipy scikit-learn matplotlib lightgbm pvlib pandapower tabulate numba` (numba makes the AC power flow much faster).
1. Put the data files in `./data` (see the *Data checklist* cell) and keep `SMOKE_TEST = False`.
2. Set `QUICK = True` for a first pass (24 days, roughly 15 min). Then set `QUICK = False` for the paper run (full year, both sites, about 1–3 h depending on the machine).
3. Everything is written to `./master_outputs/` (tables as CSV, figures as PNG at 300 dpi, `results_summary.md`, `reviewer_response_matrix.md`).

> **Integrity rule.** `SMOKE_TEST = True` generates *synthetic* files only to check that the code runs. Its numbers must never appear in a paper. The notebook stamps every output with the data mode.

In [ ]:
# ---------------------------------------------------------------- setup
import os, sys, json, time, math, glob, logging, warnings, hashlib, platform, copy
from pathlib import Path
import numpy as np
import pandas as pd
import scipy
import scipy.sparse as sp
from scipy.optimize import linprog
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    from sklearn.ensemble import HistGradientBoostingRegressor
    HAS_LGB = False
try:
    import pvlib
    HAS_PVLIB = True
except ImportError:
    HAS_PVLIB = False
import pandapower as pp
import pandapower.networks as pn

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-7s | %(message)s", force=True)
log = logging.getLogger("master")
for noisy in ["pandapower", "numba", "pvlib"]:
    logging.getLogger(noisy).setLevel(logging.ERROR)

print(f"python {platform.python_version()} | numpy {np.__version__} | pandas {pd.__version__} | scipy {scipy.__version__}")
print(f"lightgbm={HAS_LGB} | pvlib={HAS_PVLIB} | pandapower {pp.__version__}")

## Configuration
Every assumption lives here, so the paper's *Simulation setup* table can be copied from this cell.
Decisions marked **fixed a priori** must not be changed after you see results (that would be cherry-picking).

In [ ]:
class CFG:
    # ---------- run mode
    SMOKE_TEST = False          # True = synthetic files, pipeline test ONLY. Never report these numbers.
    QUICK = False               # True = 24 evaluation days (2 per month) for development
    SITES = ["caltech", "jpl"]
    OUT = Path("./master_outputs")
    DATA = Path("./data")
    SEED = 42

    # ---------- time (ALL sources converted to fixed UTC-8 = Pacific Standard Time, no DST)
    LOCAL_TZ = "Etc/GMT+8"
    STEP_MIN = 15
    DT_H = 0.25
    TRAIN_START, TRAIN_END = "2017-01-01 00:00", "2018-09-30 23:45"
    CAL_START, CAL_END = "2018-10-01 00:00", "2018-12-31 23:45"
    TEST_START, TEST_END = "2019-01-01 00:00", "2019-12-31 23:45"

    # ---------- data files (relative to DATA)
    SITE_INFO = {
        "caltech": dict(lat=34.1377, lon=-118.1253, alt=240.0,
                        nsrdb_glob="nsrdb/caltech_*.csv", acn_file="acn/caltech_2019.csv",
                        transformer_kw=150.0),
        "jpl": dict(lat=34.2013, lon=-118.1714, alt=350.0,
                    nsrdb_glob="nsrdb/jpl_*.csv", acn_file="acn/jpl_2019.csv",
                    transformer_kw=150.0),
    }
    CAISO_LOAD_GLOB = "caiso/load_*.csv"
    CAISO_LOAD_FILTERS = {"TAC_AREA_NAME": "CA ISO-TAC", "XML_DATA_ITEM": "SYS_FCST_ACT_MW"}
    CAISO_PRICE_GLOB = "caiso/lmp_*.csv"
    CAISO_PRICE_FILTERS = {"LMP_TYPE": "LMP", "MARKET_RUN_ID": "DAM", "NODE": "TH_SP15_GEN-APND"}
    ACN_CSV_TZ = "UTC"          # set from the *_CHECK.txt verdict written by acn_timeseries_to_sessions.py

    # ---------- EV / site assumptions
    EVSE_MAX_KW = 6.6           # Level-2 EVSE limit (32 A x 208 V); also caps ACN outliers
    PV_KWP = 100.0              # assumed rooftop PV at the garage (state as assumption)
    PV_DERATE = 0.85

    # ---------- network (E2)
    GARAGE_BUS = 17             # end of the IEEE 33-bus main feeder (weakest voltage)
    VMIN = 0.95                 # ANSI C84.1 Range A
    V_BASE_TARGET = 0.955       # base-load scaling: worst training-year base voltage (stressed feeder)
    LINE_RATING_FACTOR = 1.10   # branch rating = factor x max training-year base flow
    SCREEN_MARGIN = 0.01        # AC power flow is run wherever LinDistFlow min-V < VMIN + margin
    AC_VALIDATION_SAMPLES = 300

    # ---------- forecasting (E6)
    HORIZONS = {"1h": 4, "day_ahead": 96}   # steps; the simulation uses day_ahead (fixed a priori)
    QUANTILES = [0.025, 0.05, 0.95, 0.975]
    ALPHA = 0.05
    GBM_TREES = 600

    # ---------- online conformal (E5)
    CONFORMAL_MODES = ["point", "quantile", "static", "expanding", "rolling", "aci"]
    MAIN_CONFORMAL_MODE = "rolling"         # fixed a priori for the main comparison
    ROLL_DAYS = 28
    ACI_GAMMA = 0.005

    # ---------- scheduling (E3/E4)
    HMAX_STEPS = 96             # 24 h receding horizon
    EPISODE_START_HOUR = 4      # operating day = 04:00 -> 04:00
    EPISODE_MAX_H = 48
    PENALTY_UNMET = 5.0         # $/kWh not delivered (above any 2019 CAISO DAM price)
    EPS_EARLY = 1e-5            # $/kW per step of delay (tie-break: earlier is safer)
    ADMM_RHO = 0.05             # initial penalty on normalized costs (adapted by residual balancing)
    ADMM_MAX_IT = 1000
    ADMM_EPS = 1e-3             # Boyd abs/rel stopping tolerance used in simulation
    ADMM_EPS_TIGHT = 1e-4       # tighter tolerance reported in the optimality-gap study
    MAIN_METHODS = ["uncontrolled", "edf", "llf", "mpc_point", "mpc_conformal", "admm_conformal", "offline_opt"]
    PROPOSED = "admm_conformal"

    # ---------- stress tests / statistics (E2, E7)
    STRESS_V_TARGETS = [0.97, 0.96, 0.955, 0.95]
    STRESS_TR_KW = [100.0, 150.0, 200.0]
    STRESS_PENETRATION = [1.0, 1.5, 2.0]
    STRESS_METHODS = ["edf", "mpc_point", "mpc_conformal", "admm_conformal", "offline_opt"]
    STRESS_N_DAYS = 24
    N_BOOT = 5000
    BLOCK_DAYS = 7
    N_CONTEXTS_CONVERGENCE = 150


if CFG.SMOKE_TEST:
    for _s in CFG.SITE_INFO:
        CFG.SITE_INFO[_s]["acn_file"] = f"acn/{_s}_2019.json"
    CFG.OUT = Path("./master_outputs_SMOKE_TEST")
    CFG.DATA = CFG.OUT / "synthetic_data"
    CFG.QUICK = True
    CFG.GBM_TREES = 120
    CFG.N_BOOT = 500
    CFG.STRESS_N_DAYS = 3
    CFG.STRESS_V_TARGETS = [0.96, 0.95]
    CFG.STRESS_TR_KW = [100.0, 150.0]
    CFG.STRESS_PENETRATION = [1.0, 2.0]
    CFG.N_CONTEXTS_CONVERGENCE = 40
    CFG.AC_VALIDATION_SAMPLES = 60

DATA_MODE = "SYNTHETIC-SMOKE-TEST (NOT FOR PUBLICATION)" if CFG.SMOKE_TEST else "REAL DATA"
CFG.OUT.mkdir(parents=True, exist_ok=True)
np.random.seed(CFG.SEED)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 10, "axes.grid": True,
                     "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False})
PALETTE = {"uncontrolled": "#9e9e9e", "edf": "#e9c46a", "llf": "#f4a261", "mpc_point": "#e76f51",
           "mpc_conformal": "#2a9d8f", "admm_conformal": "#264653", "offline_opt": "#6a4c93"}
LABEL = {"uncontrolled": "Uncontrolled", "edf": "EDF", "llf": "LLF", "mpc_point": "MPC (point, ASA-style)",
         "mpc_conformal": "MPC (conformal)", "admm_conformal": "Proposed: ADMM + conformal",
         "offline_opt": "Offline optimum"}


def save_table(df, name, site=None):
    d = CFG.OUT / (site or "_combined")
    d.mkdir(parents=True, exist_ok=True)
    df.to_csv(d / f"{name}.csv", index=False)
    return d / f"{name}.csv"


def save_fig(fig, name, site=None):
    d = CFG.OUT / (site or "_combined")
    d.mkdir(parents=True, exist_ok=True)
    fig.suptitle((fig._suptitle.get_text() + "\n" if fig._suptitle else "") +
                 ("" if not CFG.SMOKE_TEST else "[SMOKE TEST - SYNTHETIC]"), fontsize=9)
    fig.savefig(d / f"{name}.png", bbox_inches="tight")
    plt.close(fig)


print("Data mode:", DATA_MODE, "| QUICK:", CFG.QUICK, "| outputs ->", CFG.OUT.resolve())

## Data checklist (E1)
Put these files in `./data` (filenames can differ; edit the globs in `CFG`):

| Source | What to download | Format expected |
|---|---|---|
| **ACN-Data** (you already have ACN-Data-Static) | run `python acn_timeseries_to_sessions.py --root <ACN-Data-Static> --out ./data/acn --year 2019` → `caltech_2019.csv`, `jpl_2019.csv` | CSV with `sessionID, connectionTime, disconnectTime, kWhDelivered, peak_power_kw` (API JSON also accepted) |
| **NSRDB PSM3** (nsrdb.nrel.gov or `pvlib.iotools`) | 30-min GHI and Temperature at each site's lat/lon for **2017, 2018, 2019** | standard PSM3 CSV (2 metadata rows, then `Year,Month,Day,Hour,Minute,GHI,...,Temperature`) |
| **CAISO OASIS** `SLD_FCST` | hourly system load 2017–2019 | OASIS CSV (`INTERVALSTARTTIME_GMT, TAC_AREA_NAME, XML_DATA_ITEM, MW`) |
| **CAISO OASIS** `PRC_LMP` | DAM LMP, node `TH_SP15_GEN-APND`, 2017–2019 | OASIS CSV (`INTERVALSTARTTIME_GMT, NODE, LMP_TYPE, MARKET_RUN_ID, MW`) |

The CAISO loader **refuses** to continue if timestamps are duplicated after filtering. In the old pipeline, a load series averaging about 100 GW (roughly double CAISO's record peak) points to several TAC areas being added together. This check catches that.

In [ ]:
# ---------------------------------------------------------------- synthetic files (SMOKE TEST ONLY)
def make_smoke_files():
    """Writes synthetic files in the *real* formats so the loaders are tested too."""
    rng = np.random.default_rng(7)
    d = CFG.DATA
    for sub in ["nsrdb", "caiso", "acn"]:
        (d / sub).mkdir(parents=True, exist_ok=True)
    t30 = pd.date_range("2017-01-01", "2019-12-31 23:30", freq="30min")
    loc_idx = t30.tz_localize(CFG.LOCAL_TZ)
    for site, info in CFG.SITE_INFO.items():
        if HAS_PVLIB:
            cs = pvlib.location.Location(info["lat"], info["lon"], tz=CFG.LOCAL_TZ,
                                         altitude=info["alt"]).get_clearsky(loc_idx)["ghi"].values
        else:
            cs = np.clip(1000 * np.sin(np.pi * ((t30.hour + t30.minute / 60 - 6) / 12)), 0, None)
        ndays = len(t30) // 48 + 1
        cloud_day = np.empty(ndays)
        cloud_day[0] = 0.8
        for i in range(1, ndays):
            cloud_day[i] = 0.6 * cloud_day[i - 1] + 0.4 * rng.uniform(0.3, 1.0)
        doy = np.arange(ndays) % 365
        cloud_day = np.clip(cloud_day - 0.15 * np.cos(2 * np.pi * doy / 365), 0.15, 1.0)
        k = np.repeat(cloud_day, 48)[:len(t30)] * np.clip(1 + 0.08 * rng.standard_normal(len(t30)), 0.5, 1.2)
        ghi = np.round(np.clip(cs * k, 0, None))
        temp = 18 + 7 * np.sin(2 * np.pi * (t30.dayofyear - 110) / 365) + 5 * np.sin(np.pi * (t30.hour - 8) / 12)
        for yr in [2017, 2018, 2019]:
            m = t30.year == yr
            hdr = pd.DataFrame([["NSRDB", 1, "-", "CA", "US", info["lat"], info["lon"], -8, info["alt"], -8]],
                               columns=["Source", "Location ID", "City", "State", "Country", "Latitude",
                                        "Longitude", "Time Zone", "Elevation", "Local Time Zone"])
            body = pd.DataFrame({"Year": t30[m].year, "Month": t30[m].month, "Day": t30[m].day,
                                 "Hour": t30[m].hour, "Minute": t30[m].minute, "GHI": ghi[m],
                                 "DNI": 0, "DHI": 0, "Temperature": np.round(temp[m], 1)})
            with open(d / "nsrdb" / f"{site}_{yr}.csv", "w") as f:
                hdr.to_csv(f, index=False)
                f.write(",".join(body.columns) + "\n")
                body.to_csv(f, index=False, header=False)
    # CAISO load (two TAC areas, to test the filter) and DAM LMP (two LMP types)
    th = pd.date_range("2017-01-01", "2019-12-31 23:00", freq="h", tz=CFG.LOCAL_TZ)
    hod = th.hour.values
    base = (24000 + 6000 * np.sin(np.pi * np.clip(hod - 6, 0, 16) / 16)
            + 4000 * np.sin(2 * np.pi * (th.dayofyear.values - 120) / 365) ** 2
            - 2500 * (th.dayofweek.values >= 5) + 600 * rng.standard_normal(len(th)))
    gmt = th.tz_convert("UTC").strftime("%Y-%m-%dT%H:%M:%S-00:00")
    load = pd.concat([pd.DataFrame({"INTERVALSTARTTIME_GMT": gmt, "TAC_AREA_NAME": a,
                                    "XML_DATA_ITEM": "SYS_FCST_ACT_MW", "MW": np.round(base * f, 1)})
                      for a, f in [("CA ISO-TAC", 1.0), ("PGE-TAC", 0.45)]])
    load.to_csv(d / "caiso" / "load_2017_2019.csv", index=False)
    price = 35 + 12 * np.sin(np.pi * np.clip(hod - 15, 0, 6) / 6) - 18 * np.exp(-((hod - 12) ** 2) / 8) \
        + 6 * rng.standard_normal(len(th))
    lmp = pd.concat([pd.DataFrame({"INTERVALSTARTTIME_GMT": gmt, "NODE": "TH_SP15_GEN-APND", "LMP_TYPE": t_,
                                   "MARKET_RUN_ID": "DAM", "MW": np.round(price * f, 3)})
                     for t_, f in [("LMP", 1.0), ("MCC", 0.05)]])
    lmp.to_csv(d / "caiso" / "lmp_2017_2019.csv", index=False)
    # ACN sessions, 2019 only, API-JSON format with GMT strings
    for site, wk, we, med_dwell in [("caltech", 55, 15, 7.0), ("jpl", 40, 4, 8.5)]:
        items, sid = [], 0
        for day in pd.date_range("2019-01-01", "2019-12-31", freq="D"):
            n = rng.poisson(wk if day.dayofweek < 5 else we)
            for _ in range(n):
                hr = rng.normal(8.3, 1.2) if rng.random() < 0.8 else rng.normal(13, 3)
                hr = float(np.clip(hr, 0, 23.5))
                conn = day + pd.Timedelta(hours=hr)
                dwell = float(np.clip(rng.lognormal(np.log(med_dwell), 0.5), 0.3, 30))
                kwh = float(np.clip(rng.gamma(2.2, 4.8), 0.5, 6.6 * dwell * 0.95))
                c_utc = conn.tz_localize(CFG.LOCAL_TZ).tz_convert("UTC")
                dsc = c_utc + pd.Timedelta(hours=dwell)
                items.append({"sessionID": f"{site}_{sid}", "stationID": f"S{sid % 54}",
                              "connectionTime": c_utc.strftime("%a, %d %b %Y %H:%M:%S GMT"),
                              "disconnectTime": dsc.strftime("%a, %d %b %Y %H:%M:%S GMT"),
                              "doneChargingTime": None, "kWhDelivered": round(kwh, 3), "userInputs": None})
                sid += 1
        with open(d / "acn" / f"{site}_2019.json", "w") as f:
            json.dump({"_items": items}, f)
    log.info(f"Synthetic smoke-test files written to {d}")


if CFG.SMOKE_TEST:
    make_smoke_files()

In [ ]:
# ---------------------------------------------------------------- E1 loaders
def _files(pattern):
    files = sorted(glob.glob(str(CFG.DATA / pattern)))
    if not files:
        raise FileNotFoundError(f"No files match {CFG.DATA / pattern}. See the data checklist above.")
    return files


def load_nsrdb(site):
    frames = []
    for f in _files(CFG.SITE_INFO[site]["nsrdb_glob"]):
        meta = pd.read_csv(f, nrows=1)
        tz_off = float(meta["Time Zone"].iloc[0]) if "Time Zone" in meta.columns else -8.0
        df = pd.read_csv(f, skiprows=2)
        t = pd.to_datetime(dict(year=df.Year, month=df.Month, day=df.Day, hour=df.Hour, minute=df.Minute))
        t = t - pd.to_timedelta(tz_off + 8.0, unit="h")        # file offset -> fixed UTC-8
        frames.append(pd.DataFrame({"ghi": df["GHI"].astype(float).values,
                                    "temp": df["Temperature"].astype(float).values}, index=t))
    s = pd.concat(frames).sort_index()
    s = s[~s.index.duplicated()]
    s = s.resample(f"{CFG.STEP_MIN}min").mean().interpolate(limit=4)
    return s


def load_oasis(pattern, filters, kind):
    df = pd.concat([pd.read_csv(f) for f in _files(pattern)], ignore_index=True)
    df.columns = [c.strip() for c in df.columns]
    for k, v in filters.items():
        if k in df.columns:
            df = df[df[k] == v]
    tcol = "INTERVALSTARTTIME_GMT" if "INTERVALSTARTTIME_GMT" in df.columns else \
        [c for c in df.columns if "time" in c.lower()][0]
    vcol = next(c for c in ["MW", "VALUE", "LOAD_MW", "PRC"] if c in df.columns)
    t = pd.to_datetime(df[tcol], utc=True).dt.tz_convert(CFG.LOCAL_TZ).dt.tz_localize(None)
    s = pd.Series(df[vcol].astype(float).values, index=t.values).sort_index()
    ndup = int(s.index.duplicated().sum())
    if ndup:
        info = {c: df[c].unique()[:8].tolist() for c in df.columns if df[c].dtype == object and c != tcol}
        raise ValueError(f"{kind}: {ndup} duplicate timestamps after filtering {filters}. "
                         f"Add filters so ONE series remains. Columns/values seen: {info}")
    s = s.resample(f"{CFG.STEP_MIN}min").mean()
    return s.interpolate(limit=8) if kind == "load" else s.ffill(limit=8)


def load_acn(site):
    path = CFG.DATA / CFG.SITE_INFO[site]["acn_file"]
    if path.suffix.lower() == ".json":
        obj = json.load(open(path))
        df = pd.DataFrame(obj["_items"] if isinstance(obj, dict) else obj)
        conv = lambda c: pd.to_datetime(df[c], utc=True).dt.tz_convert(CFG.LOCAL_TZ).dt.tz_localize(None)
    else:
        df = pd.read_csv(path)

        def conv(c):
            t = pd.to_datetime(df[c], errors="coerce")
            if t.dt.tz is None:
                t = t.dt.tz_localize(CFG.ACN_CSV_TZ, ambiguous="NaT", nonexistent="NaT")
            return t.dt.tz_convert(CFG.LOCAL_TZ).dt.tz_localize(None)
    out = pd.DataFrame({"session_id": df["sessionID"].astype(str),
                        "conn": conv("connectionTime"), "disc": conv("disconnectTime"),
                        "kwh": pd.to_numeric(df["kWhDelivered"], errors="coerce")})
    if "peak_power_kw" in df.columns:
        out["peak_kw"] = pd.to_numeric(df["peak_power_kw"], errors="coerce")
    return out.dropna(subset=["conn", "disc", "kwh"])


def build_site_frame(site):
    """One 15-min frame from TRAIN_START to TEST_END with every input series."""
    idx = pd.date_range(CFG.TRAIN_START, CFG.TEST_END, freq=f"{CFG.STEP_MIN}min")
    sol = load_nsrdb(site).reindex(idx).interpolate(limit=8)
    load = load_oasis(CFG.CAISO_LOAD_GLOB, CFG.CAISO_LOAD_FILTERS, "load").reindex(idx).interpolate(limit=8)
    price = load_oasis(CFG.CAISO_PRICE_GLOB, CFG.CAISO_PRICE_FILTERS, "price").reindex(idx).ffill(limit=8)
    F = pd.DataFrame({"ghi": sol["ghi"], "temp": sol["temp"], "load_mw": load, "price": price}, index=idx)
    miss = F.isna().mean()
    if (miss > 0.02).any():
        log.warning(f"[{site}] missing share per column:\n{miss}")
    F[["ghi", "temp"]] = F[["ghi", "temp"]].interpolate(limit_direction="both")
    F["load_mw"] = F["load_mw"].interpolate(limit_direction="both")
    F["price"] = F["price"].ffill().bfill()
    F["ghi"] = F["ghi"].clip(lower=0)
    return F

In [ ]:
# ---------------------------------------------------------------- E1 sessions on the real calendar
def prepare_sessions(site, grid):
    raw = load_acn(site)
    n_raw = len(raw)
    s = raw[(raw.conn >= grid[0]) & (raw.conn <= grid[-1])].copy()
    s["dwell_h"] = (s.disc - s.conn).dt.total_seconds() / 3600
    s = s[(s.dwell_h >= 0.25) & (s.kwh > 0.1)].copy()
    step = pd.Timedelta(minutes=CFG.STEP_MIN)
    s["a"] = np.ceil((s.conn - grid[0]) / step).astype(int)
    s["d"] = np.floor((s.disc - grid[0]) / step).astype(int).clip(upper=len(grid) - 1)
    s = s[s.d > s.a].copy()
    if "peak_kw" in s.columns:
        s["rmax"] = np.where(s.peak_kw > 0.5, np.minimum(CFG.EVSE_MAX_KW, s.peak_kw * 1.05), CFG.EVSE_MAX_KW)
    else:
        s["rmax"] = CFG.EVSE_MAX_KW
    s["e_feas"] = s.rmax * (s.d - s.a) * CFG.DT_H
    s["clipped"] = s.kwh > s.e_feas + 1e-9
    s["e"] = np.minimum(s.kwh, s.e_feas)
    s = s.sort_values("a").reset_index(drop=True)
    # episode = operating day starting at EPISODE_START_HOUR
    s["day"] = (s.conn - pd.Timedelta(hours=CFG.EPISODE_START_HOUR)).dt.normalize()
    conc = np.zeros(len(grid) + 1, int)
    np.add.at(conc, s.a.values, 1)
    np.add.at(conc, s.d.values, -1)
    conc = np.cumsum(conc)[:-1]
    days = s.groupby("day").size()
    stats_ = {"site": site, "sessions_raw_file": n_raw, "sessions_in_test_year": len(s),
              "days_with_sessions": int(days.size), "sessions_per_day_mean": round(days.mean(), 1),
              "sessions_per_weekday_mean": round(days[days.index.dayofweek < 5].mean(), 1),
              "peak_concurrent_EVs": int(conc.max()), "energy_kwh_total": round(s.e.sum(), 1),
              "energy_kwh_mean": round(s.e.mean(), 2), "dwell_h_median": round(s.dwell_h.median(), 2),
              "share_energy_clipped_to_feasible_pct": round(100 * s.clipped.mean(), 2),
              "first_arrival": str(s.conn.min()), "last_arrival": str(s.conn.max())}
    return s, stats_, conc

In [ ]:
# ---------------------------------------------------------------- E6 solar geometry and features
def solar_geometry(index, site):
    info = CFG.SITE_INFO[site]
    if HAS_PVLIB:
        loc = pvlib.location.Location(info["lat"], info["lon"], tz=CFG.LOCAL_TZ, altitude=info["alt"])
        idx = index.tz_localize(CFG.LOCAL_TZ)
        spos = loc.get_solarposition(idx)
        cs = loc.get_clearsky(idx, model="ineichen", solar_position=spos)
        return pd.DataFrame({"ghi_cs": cs["ghi"].values, "zenith": spos["apparent_zenith"].values}, index=index)
    # fallback: simple solar geometry + Haurwitz clear-sky model
    doy = index.dayofyear.values
    decl = np.deg2rad(23.45) * np.sin(2 * np.pi * (284 + doy) / 365)
    b = 2 * np.pi * (doy - 81) / 364
    eot = 9.87 * np.sin(2 * b) - 7.53 * np.cos(b) - 1.5 * np.sin(b)
    solar_time = index.hour.values + index.minute.values / 60 + (4 * (info["lon"] + 120) + eot) / 60
    ha = np.deg2rad(15 * (solar_time - 12))
    lat = np.deg2rad(info["lat"])
    cosz = np.sin(lat) * np.sin(decl) + np.cos(lat) * np.cos(decl) * np.cos(ha)
    zen = np.rad2deg(np.arccos(np.clip(cosz, -1, 1)))
    ghi_cs = np.where(cosz > 0, 1098 * cosz * np.exp(-0.057 / np.maximum(cosz, 1e-3)), 0)
    return pd.DataFrame({"ghi_cs": ghi_cs, "zenith": zen}, index=index)


def lag_list(h):
    day = 96 * int(np.ceil(h / 96))
    return sorted(set([h, h + 1, h + 2, h + 4, day, day + 96, max(672, day)]))


def build_features(F, target, h):
    """All lagged inputs have lag >= h (asserted). 'Now' inputs are deterministic (calendar, clear-sky)."""
    X, lag_used = {}, []
    idx = F.index
    hr = idx.hour.values + idx.minute.values / 60
    X["hour_sin"], X["hour_cos"] = np.sin(2 * np.pi * hr / 24), np.cos(2 * np.pi * hr / 24)
    X["doy_sin"], X["doy_cos"] = np.sin(2 * np.pi * idx.dayofyear / 365.25), np.cos(2 * np.pi * idx.dayofyear / 365.25)
    X["ghi_cs"] = F["ghi_cs"].values
    X["cos_zen"] = np.cos(np.deg2rad(F["zenith"].values))
    day = 96 * int(np.ceil(h / 96))
    if target == "pv":
        lag_cols = {"kc": F["kc"], "ghi": F["ghi"], "temp": F["temp"]}
        roll = F["kc"].rolling(96, min_periods=8).mean()
    else:
        from pandas.tseries.holiday import USFederalHolidayCalendar
        hol = USFederalHolidayCalendar().holidays(idx.min(), idx.max())
        X["dow"] = idx.dayofweek.values
        X["holiday"] = idx.normalize().isin(hol).astype(int)
        lag_cols = {"load": F["load_mw"], "temp": F["temp"]}
        roll = F["load_mw"].rolling(96, min_periods=48).mean()
    for name, ser in lag_cols.items():
        for L in lag_list(h):
            X[f"{name}_lag{L}"] = ser.shift(L).values
            lag_used.append(L)
    X[f"roll_day_lag{day}"] = roll.shift(day).values
    lag_used.append(day)
    assert min(lag_used) >= h, "LEAKAGE: a lagged feature is closer than the forecast horizon"
    return pd.DataFrame(X, index=idx), min(lag_used)


def _gbm(objective, alpha=None):
    if HAS_LGB:
        kw = dict(n_estimators=CFG.GBM_TREES, learning_rate=0.03, num_leaves=63, min_child_samples=50,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=CFG.SEED, verbose=-1)
        return lgb.LGBMRegressor(objective=objective, alpha=alpha, **kw) if alpha else \
            lgb.LGBMRegressor(objective=objective, **kw)
    loss = "quantile" if alpha else "squared_error"
    return HistGradientBoostingRegressor(loss=loss, quantile=alpha, max_iter=CFG.GBM_TREES,
                                         learning_rate=0.05, random_state=CFG.SEED)


def winkler(y, lo, hi, alpha):
    w = hi - lo
    return np.mean(w + (2 / alpha) * (lo - y) * (y < lo) + (2 / alpha) * (y - hi) * (y > hi))


def conformal_q(scores, alpha):
    s = np.sort(np.asarray(scores)[np.isfinite(scores)])
    n = len(s)
    if n == 0:
        return 0.0
    k = min(n - 1, int(np.ceil((n + 1) * (1 - alpha))) - 1)
    return float(s[max(k, 0)])

In [ ]:
# ---------------------------------------------------------------- E6 training and evaluation
def run_forecasting(site, F):
    t0 = time.time()
    geo = solar_geometry(F.index, site)
    F = F.join(geo)
    F["kc"] = np.where(F.ghi_cs > 20, F.ghi / F.ghi_cs.where(F.ghi_cs > 20), np.nan)
    F["kc"] = F["kc"].clip(0, 1.5)
    tr = (F.index >= CFG.TRAIN_START) & (F.index <= CFG.TRAIN_END)
    ca = (F.index >= CFG.CAL_START) & (F.index <= CFG.CAL_END)
    te = (F.index >= CFG.TEST_START) & (F.index <= CFG.TEST_END)
    assert F.index[tr].max() < F.index[ca].min() and F.index[ca].max() < F.index[te].min(), "periods overlap"
    day_mask = F.ghi_cs.values > 20
    rows, out, leak = [], {}, []
    for target in ["pv", "load"]:
        y_all = F["ghi"] if target == "pv" else F["load_mw"]
        for hname, h in CFG.HORIZONS.items():
            X, minlag = build_features(F, target, h)
            leak.append({"target": target, "horizon": hname, "horizon_steps": h, "min_feature_lag_steps": minlag,
                         "n_features": X.shape[1], "ok": minlag >= h})
            fit_m = tr & y_all.notna().values & (day_mask if target == "pv" else True)
            preds = {}
            m = _gbm("regression").fit(X[fit_m], y_all[fit_m])
            preds["point"] = m.predict(X)
            for q in CFG.QUANTILES:
                mq = _gbm("quantile", q).fit(X[fit_m], y_all[fit_m])
                preds[q] = mq.predict(X)
            if target == "pv":                      # physical correction: no sun -> 0, never negative
                for k_ in preds:
                    preds[k_] = np.where(day_mask, np.clip(preds[k_], 0, None), 0.0)
            P = pd.DataFrame({k_ if isinstance(k_, str) else f"q{k_}": v for k_, v in preds.items()}, index=F.index)
            P[["q0.025", "q0.05", "q0.95", "q0.975"]] = np.sort(P[["q0.025", "q0.05", "q0.95", "q0.975"]].values, axis=1)
            dlag = 96 * int(np.ceil(h / 96))
            if target == "pv":
                kc_ref = F["kc"].shift(h).where(F["kc"].shift(h).notna(), F["kc"].shift(dlag))
                base = (kc_ref.fillna(F["kc"].shift(dlag)).fillna(0.8) * F.ghi_cs).where(day_mask, 0).values
                base_name = "smart persistence"
            else:
                base = F["load_mw"].shift(dlag).values
                base_name = "day persistence"
            # CQR (two-sided, 1-ALPHA) calibrated on the calibration period
            ev_c = ca & y_all.notna().values & (day_mask if target == "pv" else True)
            s_two = np.maximum(P["q0.025"].values - y_all.values, y_all.values - P["q0.975"].values)
            q_two = conformal_q(s_two[ev_c], CFG.ALPHA)
            lo_c = P["q0.025"].values - q_two
            hi_c = P["q0.975"].values + q_two
            if target == "pv":
                lo_c = np.where(day_mask, np.clip(lo_c, 0, None), 0)
                hi_c = np.where(day_mask, hi_c, 0)
            ev = te & y_all.notna().values & (day_mask if target == "pv" else True)
            y = y_all.values[ev]
            rmse_b = np.sqrt(np.nanmean((base[ev] - y) ** 2))
            for mname, yhat in [("LightGBM" if HAS_LGB else "HistGBM", P["point"].values[ev]), (base_name, base[ev])]:
                ok = np.isfinite(yhat)
                rmse = np.sqrt(np.mean((yhat[ok] - y[ok]) ** 2))
                rows.append({"target": target, "horizon": hname, "model": mname,
                             "eval_rows": "daytime only" if target == "pv" else "all",
                             "RMSE": rmse, "MAE": np.mean(np.abs(yhat[ok] - y[ok])),
                             "MBE": np.mean(yhat[ok] - y[ok]), "nRMSE_pct": 100 * rmse / np.mean(y[ok]),
                             "skill_vs_persistence": 1 - rmse / rmse_b, "R2": 1 - np.sum((yhat[ok] - y[ok]) ** 2) / np.sum((y[ok] - y[ok].mean()) ** 2)})
            for iname, lo, hi in [("raw quantiles", P["q0.025"].values, P["q0.975"].values), ("CQR", lo_c, hi_c)]:
                rows.append({"target": target, "horizon": hname, "model": f"interval: {iname}",
                             "coverage_pct": 100 * np.mean((y >= lo[ev]) & (y <= hi[ev])),
                             "mean_width": np.mean(hi[ev] - lo[ev]),
                             "winkler": winkler(y, lo[ev], hi[ev], CFG.ALPHA)})
            out[(target, hname)] = dict(P=P, lo_c=lo_c, hi_c=hi_c, q_two=q_two, base=base)
    metrics = pd.DataFrame(rows)
    log.info(f"[{site}] forecasting done in {time.time() - t0:.0f}s")
    return F, out, metrics, pd.DataFrame(leak)

## E2: Distribution network
* **Topology:** IEEE 33-bus (Baran & Wu) from pandapower; the garage is connected at bus 17 (the end of the main feeder).
* **Base load:** each bus's nominal load × a scale × the *normalized* CAISO load shape (load / training-year maximum). Normalizing removes any dependence on the absolute CAISO level.
* **Scale:** chosen so the worst **training-year** base voltage equals `V_BASE_TARGET`, i.e. a feeder with limited headroom. The stress test sweeps this value.
* **Scheduling constraints (identical for every controlled method):** LinDistFlow voltage limits at all 33 buses, ratings of the branches on the garage path (sized for the training-year peak plus the garage at its transformer rating, ×`LINE_RATING_FACTOR`), and the site transformer. A **loss margin** calibrated against AC power flow is added, because LinDistFlow ignores losses.
* **Evaluation:** full **AC power flow** (pandapower Newton-Raphson) at every timestep where LinDistFlow is within `SCREEN_MARGIN` of the limit. The linearization error elsewhere is measured and reported.

In [ ]:
class Feeder:
    def __init__(self, base_shape_train, v_target=CFG.V_BASE_TARGET, tr_kw=150.0, garage_bus=CFG.GARAGE_BUS):
        net = pn.case33bw()
        net.line = net.line[net.line.in_service].copy()
        self.net = net
        self.g = garage_bus
        self.tr_kw = tr_kw
        vn = float(net.bus.vn_kv.iloc[0])
        nb = len(net.bus)
        L = net.line
        self.r = (L.r_ohm_per_km * L.length_km).values / vn ** 2     # p.u. on 1 MVA base
        self.x = (L.x_ohm_per_km * L.length_km).values / vn ** 2
        children = {b: [] for b in range(nb)}
        for fb, tb in zip(L.from_bus.values, L.to_bus.values):
            children[int(fb)].append(int(tb))
        def subtree(b):
            out, stack = [], [b]
            while stack:
                u = stack.pop(); out.append(u); stack.extend(children[u])
            return out
        self.D = np.zeros((len(L), nb))
        for li, tb in enumerate(L.to_bus.values):
            self.D[li, subtree(int(tb))] = 1.0
        self.Dg = self.D[:, self.g]                                    # lines on the garage path
        self.s = (self.D * (self.Dg * 2 * self.r)[:, None]).sum(0)      # dV^2 per MW at garage
        self.Pn = np.zeros(nb); self.Qn = np.zeros(nb)
        np.add.at(self.Pn, net.load.bus.values, net.load.p_mw.values)
        np.add.at(self.Qn, net.load.bus.values, net.load.q_mvar.values)
        # scale base load so the worst training-year base voltage equals v_target
        k = self._v2(np.max(base_shape_train), 1.0)[0]
        self.scale = float((1 - v_target ** 2) / (1 - k.min()))
        # branch ratings: sized for the training-year peak base flow PLUS the garage at its transformer
        # rating (the feeder was built to host the garage), times LINE_RATING_FACTOR
        Pl, Ql = self._flows(base_shape_train)
        Pl = Pl + (tr_kw / 1000) * self.Dg[None, :]
        self.S_rating = CFG.LINE_RATING_FACTOR * np.max(np.sqrt(Pl ** 2 + Ql ** 2), axis=0)
        self.garage_load = pp.create_load(net, bus=self.g, p_mw=0.0, q_mvar=0.0, name="garage")
        self.margin = 0.0
        self.margin = self._calibrate_margin(base_shape_train)

    def _flows(self, shape, scale=None):
        sc = self.scale if scale is None else scale
        shape = np.atleast_1d(shape)
        Pl = (sc * shape[:, None] * self.Pn[None, :]) @ self.D.T
        Ql = (sc * shape[:, None] * self.Qn[None, :]) @ self.D.T
        return Pl, Ql

    def _v2(self, shape, scale=None, garage_mw=0.0):
        Pl, Ql = self._flows(shape, scale)
        Pl = Pl + np.atleast_1d(garage_mw)[:, None] * self.Dg[None, :]
        return 1.0 - (2 * (Pl * self.r + Ql * self.x)) @ self.D

    def ev_cap_kw(self, shape, pv_kw, background_kw=0.0):
        """Max EV charging (kW) per step so that voltage, branch ratings and transformer hold (linearized)."""
        V2 = self._v2(shape)
        sens = self.s > 1e-12
        vcap = np.min((V2[:, sens] - (CFG.VMIN ** 2 + self.margin)) / self.s[sens], axis=1)
        Pl, Ql = self._flows(shape)
        gl = self.Dg > 0
        lcap = np.min(np.sqrt(np.maximum(self.S_rating[gl] ** 2 - Ql[:, gl] ** 2, 0)) - Pl[:, gl], axis=1)
        net_cap_kw = np.minimum(np.minimum(vcap, lcap) * 1000, self.tr_kw)
        return np.maximum(0.0, net_cap_kw + np.asarray(pv_kw) - np.asarray(background_kw))

    def lin_eval(self, shape, garage_kw):
        V2 = self._v2(shape, garage_mw=np.asarray(garage_kw) / 1000)
        V2b = self._v2(shape)
        Pl, Ql = self._flows(shape)
        Pl = Pl + (np.asarray(garage_kw) / 1000)[:, None] * self.Dg[None, :]
        load_pct = 100 * np.max(np.sqrt(Pl ** 2 + Ql ** 2)[:, self.Dg > 0] / self.S_rating[self.Dg > 0], axis=1)
        return np.sqrt(np.clip(V2.min(1), 0, None)), np.sqrt(np.clip(V2b.min(1), 0, None)), load_pct

    def ac_eval(self, shape_t, garage_kw):
        net = self.net
        lo = net.load.index != self.garage_load
        net.load.loc[lo, "p_mw"] = self.scale * shape_t * self.Pn[net.load.loc[lo, "bus"].values]
        net.load.loc[lo, "q_mvar"] = self.scale * shape_t * self.Qn[net.load.loc[lo, "bus"].values]
        net.load.loc[self.garage_load, "p_mw"] = garage_kw / 1000
        pp.runpp(net, numba=False, init="flat", calculate_voltage_angles=False)
        S = np.sqrt(net.res_line.p_from_mw.values ** 2 + net.res_line.q_from_mvar.values ** 2)
        gl = self.Dg > 0
        return float(net.res_bus.vm_pu.min()), float(100 * np.max(S[gl] / self.S_rating[gl]))

    def _calibrate_margin(self, shape_train, n=60):
        rng = np.random.default_rng(CFG.SEED)
        hi = np.quantile(shape_train, 0.8)
        cand = shape_train[shape_train >= hi]
        errs = []
        for sh in rng.choice(cand, size=min(n, len(cand)), replace=False):
            g_kw = rng.uniform(0, self.tr_kw)
            v_ac, _ = self.ac_eval(sh, g_kw)
            v_lin2 = self._v2(np.array([sh]), garage_mw=g_kw / 1000).min()
            errs.append(v_lin2 - v_ac ** 2)                 # >0 means LinDistFlow is optimistic
        return float(max(0.0, np.quantile(errs, 0.95)))

## E3 / E4: Schedulers
All controlled methods get **the same information at each 15-min step**: the EVs currently plugged in (energy still needed, departure time, maximum rate), the DAM prices for the horizon (published the day before), the PV forecast, and the EV capacity limit from the network model. Only *uncontrolled* ignores the limit, and only the *offline optimum* sees the future (actual load, actual PV, all arrivals of the day).

**Proposed protocol (ADMM sharing, Boyd et al. 2011, §7.3).**
Each EV agent *i* solves a private problem: it charges its own energy by its own deadline within its own rate limit. Departure time and energy demand never leave the agent. The coordinator (site/DSO agent) owns the shared cost: grid import at the DAM price minus on-site PV, plus the network limit from the conformal forecast. Each iteration, every EV uploads its tentative profile and the coordinator broadcasts one vector, the scaled congestion price. The problem is convex, so ADMM converges to the centralized optimum. A final proportional projection guarantees the step is feasible even if iterations stop early.
In the old paper's terms: the EV agents are the *EV agents*; the coordinator merges the *Grid*, *Solar*, *Price* and *Charger* roles. The congestion price learned online replaces the hand-tuned weight `w_g` (old Mechanism B).

In [ ]:
def project_step(r0, rmax, e_rem, cap0):
    r0 = np.clip(r0, 0, np.minimum(rmax, e_rem / CFG.DT_H))
    tot = r0.sum()
    if cap0 is not None and tot > cap0 + 1e-9:
        r0 = r0 * (cap0 / tot)
    return r0


def sched_uncontrolled(ctx):
    return np.minimum(ctx["rmax"], ctx["e"] / CFG.DT_H), {}


def _greedy(order, ctx):
    r = np.zeros(len(ctx["e"]))
    room = ctx["cap"][0]
    for i in order:
        r[i] = min(ctx["rmax"][i], ctx["e"][i] / CFG.DT_H, max(room, 0))
        room -= r[i]
    return r, {}


def sched_edf(ctx):
    return _greedy(np.argsort(ctx["d"], kind="stable"), ctx)


def sched_llf(ctx):
    lax = ctx["d"] * CFG.DT_H - ctx["e"] / np.maximum(ctx["rmax"], 1e-6)
    return _greedy(np.argsort(lax, kind="stable"), ctx)


def build_lp(e, a, d, rmax, cap, price, pv):
    """min sum_k c_k g_k + eps*k*sum_i r_ik + M*u_i  s.t. sum_k r_ik dt + u_i = e_i,
       sum_i r_ik <= cap_k, sum_i r_ik - g_k <= pv_k, 0<=r<=rmax, g,u>=0  (sessions available a_i <= k < d_i)."""
    N, H = len(e), len(cap)
    ii, kk = [], []
    for i in range(N):
        ks = np.arange(max(a[i], 0), min(d[i], H))
        ii.append(np.full(len(ks), i)); kk.append(ks)
    ii, kk = np.concatenate(ii).astype(int), np.concatenate(kk).astype(int)
    nr = len(ii)
    dt = CFG.DT_H
    c = np.concatenate([CFG.EPS_EARLY * kk * dt, np.maximum(price, 0) / 1000 * dt, np.full(N, CFG.PENALTY_UNMET)])
    Aeq = sp.hstack([sp.csr_matrix((np.full(nr, dt), (ii, np.arange(nr))), shape=(N, nr)),
                     sp.csr_matrix((N, H)), sp.identity(N, format="csr")]).tocsr()
    R = sp.csr_matrix((np.ones(nr), (kk, np.arange(nr))), shape=(H, nr))
    Aub = sp.vstack([sp.hstack([R, sp.csr_matrix((H, H)), sp.csr_matrix((H, N))]),
                     sp.hstack([R, -sp.identity(H), sp.csr_matrix((H, N))])]).tocsr()
    bub = np.concatenate([cap, pv])
    bounds = np.vstack([np.column_stack([np.zeros(nr), rmax[ii]]),
                        np.column_stack([np.zeros(H), np.full(H, np.inf)]),
                        np.column_stack([np.zeros(N), np.full(N, np.inf)])])
    res = linprog(c, A_ub=Aub, b_ub=bub, A_eq=Aeq, b_eq=e, bounds=bounds, method="highs")
    X = np.zeros((N, H))
    if res.status == 0:
        X[ii, kk] = res.x[:nr]
    return X, res.status == 0, (res.fun if res.status == 0 else np.nan)


def lp_objective(X, e, cap, price, pv):
    agg = X.sum(0)
    dt = CFG.DT_H
    unmet = np.maximum(e - X.sum(1) * dt, 0)
    return float(np.sum(np.maximum(price, 0) / 1000 * dt * np.maximum(agg - pv, 0))
                 + np.sum(CFG.EPS_EARLY * np.arange(len(cap)) * dt * agg) + CFG.PENALTY_UNMET * unmet.sum())


def sched_mpc(ctx):
    N = len(ctx["e"])
    X, ok, _ = build_lp(ctx["e"], np.zeros(N, int), ctx["d"], ctx["rmax"], ctx["cap"], ctx["price"], ctx["pv"])
    if not ok:
        return sched_edf(ctx)[0], {"lp_fail": 1}
    return X[:, 0], {}


def admm_solve(e, d, rmax, cap, price, pv, x0=None, rho=None, max_it=None, eps=None, trace=False):
    """ADMM sharing problem (Boyd et al. 2011, Sec. 7.3), costs normalized to O(1), Boyd stopping rule
    (Sec. 3.3.1) and residual balancing (Sec. 3.4.1).
    Agent i (private e_i, d_i, rmax_i): min sum_k a_k x_ik  s.t. 0 <= x_ik <= rmax_i (k < d_i), sum_k x_ik dt <= e_i
    Coordinator: g(s) = c_k * max(0, s_k - pv_k) + indicator(0 <= s_k <= cap_k), s = sum_i x_i."""
    rho = CFG.ADMM_RHO if rho is None else rho
    max_it = CFG.ADMM_MAX_IT if max_it is None else max_it
    eps = CFG.ADMM_EPS if eps is None else eps
    N, H = len(e), len(cap)
    dt = CFG.DT_H
    ub = rmax[:, None] * (np.arange(H)[None, :] < d[:, None])
    c = np.maximum(price, 0) / 1000 * dt
    scale = max(float(c.max()), 1e-6)
    c = c / scale
    a = (CFG.EPS_EARLY * np.arange(H) - CFG.PENALTY_UNMET) * dt / scale
    x = np.zeros((N, H)) if x0 is None else np.clip(x0, 0, ub)
    xbar = x.mean(0)
    zbar = np.clip(x.sum(0), 0, cap) / N
    w = np.zeros(H)
    hist = []
    it = 0
    for it in range(1, max_it + 1):
        v = x - xbar + zbar - w
        # ---- EV agents (in parallel, private data): closed form + bisection on the energy multiplier
        x = np.clip(v - a / rho, 0, ub)
        over = x.sum(1) * dt > e + 1e-9
        if over.any():
            vo, uo, eo = v[over], ub[over], e[over]
            lo = np.zeros(over.sum())
            hi = np.full(over.sum(), (np.abs(vo).max() * rho + np.abs(a).max()) / dt + 1)
            for _ in range(45):
                mu = 0.5 * (lo + hi)
                too = np.clip(vo - (a[None, :] + mu[:, None] * dt) / rho, 0, uo).sum(1) * dt > eo
                lo = np.where(too, mu, lo); hi = np.where(too, hi, mu)
            x[over] = np.clip(vo - (a[None, :] + hi[:, None] * dt) / rho, 0, uo)
        xbar = x.mean(0)
        # ---- coordinator: prox of import cost + network limit, separable per step
        s = N * (w + xbar)
        s = np.where(s <= pv, s, np.maximum(pv, s - c * N / rho))
        z_new = np.clip(s, 0, cap) / N
        w = w + xbar - z_new
        r_pri = np.sqrt(N) * np.linalg.norm(xbar - z_new)
        r_dual = rho * np.sqrt(N) * np.linalg.norm(z_new - zbar)
        zbar = z_new
        e_pri = np.sqrt(N * H) * eps * CFG.EVSE_MAX_KW + eps * max(np.linalg.norm(x), np.sqrt(N) * np.linalg.norm(zbar))
        e_dual = np.sqrt(N * H) * eps + eps * rho * np.sqrt(N) * np.linalg.norm(w)
        if trace:
            hist.append((it, r_pri, r_dual))
        if r_pri < e_pri and r_dual < e_dual:
            break
        if it % 10 == 0:
            if r_pri > 10 * r_dual:
                rho *= 2; w /= 2
            elif r_dual > 10 * r_pri:
                rho /= 2; w *= 2
    return x, {"iters": it, "uplink_floats": it * N * H, "downlink_floats": it * H, "hist": hist}


_ADMM_WARM = {}


def sched_admm(ctx):
    ids = ctx["ids"]
    H = len(ctx["cap"])
    x0 = np.zeros((len(ids), H))
    for j, sid in enumerate(ids):
        prev = _ADMM_WARM.get(sid)
        if prev is not None:
            m = min(H, len(prev) - 1)
            x0[j, :m] = prev[1:m + 1]
    X, info = admm_solve(ctx["e"], ctx["d"], ctx["rmax"], ctx["cap"], ctx["price"], ctx["pv"], x0=x0)
    for j, sid in enumerate(ids):
        _ADMM_WARM[sid] = X[j]
    return X[:, 0], info


SCHEDULERS = {"uncontrolled": sched_uncontrolled, "edf": sched_edf, "llf": sched_llf,
              "mpc_point": sched_mpc, "mpc_conformal": sched_mpc, "admm_conformal": sched_admm}
METHOD_MODE = {"mpc_point": "point"}       # all other controlled online methods use MAIN_CONFORMAL_MODE

## E5: Online conformal recalibration (the closed loop)
The network limit is computed from a **one-sided** bound: base load ≤ upper bound (high load lowers voltage) and PV ≥ lower bound (less PV means more grid import).
Nonconformity scores, recomputed every midnight from realized data (independent of the schedules):
`s_load = y − Q̂0.95` and `s_pv = Q̂0.05 − y` (daytime only). The bound is `Q̂ ± q`, where *q* comes from:

| mode | q |
|---|---|
| point | no bound: the point forecast is used (what most papers do) |
| quantile | 0: raw quantile model, uncalibrated |
| static | split conformal on the calibration period (Oct–Dec 2018), then frozen |
| expanding | calibration scores + all realized scores (the old Mechanism A) |
| rolling | the last `ROLL_DAYS` days of realized scores (**main mode, fixed a priori**) |
| aci | adaptive conformal inference (Gibbs & Candès 2021): α updated every step |

In [ ]:
def online_margins(scores, cal_scores, days, mode, alpha=CFG.ALPHA):
    """scores: realized test-period scores (T,), days: day id per step. Returns q per unique day (dict)."""
    uniq = pd.unique(days)
    qd = {}
    pool = list(cal_scores[np.isfinite(cal_scores)])
    q_static = conformal_q(np.asarray(pool), alpha)
    by_day = {dd: scores[days == dd] for dd in uniq}
    hist_days = []
    a_t = alpha
    for dd in uniq:
        if mode in ("point", "quantile"):
            qd[dd] = 0.0
        elif mode == "static":
            qd[dd] = q_static
        elif mode == "expanding":
            qd[dd] = conformal_q(np.asarray(pool), alpha)
        elif mode == "rolling":
            recent = np.concatenate(hist_days[-CFG.ROLL_DAYS:]) if hist_days else np.asarray(pool[-96 * CFG.ROLL_DAYS:])
            qd[dd] = conformal_q(recent, alpha)
        elif mode == "aci":
            recent = np.concatenate(hist_days[-CFG.ROLL_DAYS:]) if hist_days else np.asarray(pool[-96 * CFG.ROLL_DAYS:])
            recent = recent[np.isfinite(recent)]
            qd[dd] = float(np.quantile(recent, 1 - np.clip(a_t, 0.001, 0.5))) if len(recent) else q_static
        sd = by_day[dd][np.isfinite(by_day[dd])]
        if mode == "aci":
            for sc in sd:                                    # per-step alpha update
                a_t = a_t + CFG.ACI_GAMMA * (alpha - float(sc > qd[dd]))
        hist_days.append(sd)
        if mode == "expanding":
            pool.extend(sd.tolist())
    return qd

In [ ]:
# ---------------------------------------------------------------- simulation engine
def site_arrays(site, F, fc):
    """Test-period arrays: actual & forecast base-load shape, PV kW, price, one-sided bounds."""
    te = (F.index >= CFG.TEST_START) & (F.index <= CFG.TEST_END)
    ca = (F.index >= CFG.CAL_START) & (F.index <= CFG.CAL_END)
    tr = (F.index >= CFG.TRAIN_START) & (F.index <= CFG.TRAIN_END)
    norm = float(F.load_mw[tr].max())                       # training-year maximum: no test information
    kw = CFG.PV_KWP / 1000 * CFG.PV_DERATE
    Pl, Pp = fc[("load", "day_ahead")]["P"], fc[("pv", "day_ahead")]["P"]
    A = {"grid": F.index[te], "norm": norm,
         "shape_act": F.load_mw.values / norm, "shape_pt": Pl["point"].values / norm,
         "shape_q95": Pl["q0.95"].values / norm, "pv_act": F.ghi.values * kw, "pv_pt": Pp["point"].values * kw,
         "pv_q05": Pp["q0.05"].values * kw, "price": F.price.values, "day_mask": F.ghi_cs.values > 20}
    s_load = A["shape_act"] - A["shape_q95"]
    s_pv = np.where(A["day_mask"], A["pv_q05"] - A["pv_act"], np.nan)
    A["s_load_cal"], A["s_pv_cal"] = s_load[ca], s_pv[ca]
    for k_ in ["shape_act", "shape_pt", "shape_q95", "pv_act", "pv_pt", "pv_q05", "price", "day_mask"]:
        A[k_] = A[k_][te]
    A["s_load"], A["s_pv"] = s_load[te], s_pv[te]
    A["shape_train"] = F.load_mw.values[tr] / norm
    A["day_id"] = (A["grid"] - pd.Timedelta(hours=CFG.EPISODE_START_HOUR)).normalize()
    A["q"] = {m: (online_margins(A["s_load"], A["s_load_cal"], A["day_id"].values, m),
                  online_margins(A["s_pv"], A["s_pv_cal"], A["day_id"].values, m)) for m in CFG.CONFORMAL_MODES}
    return A


def bounds_for(A, mode, t, H):
    """Forecast bounds seen at decision time t for steps t..t+H-1 (q of the decision day)."""
    sl = slice(t, t + H)
    if mode == "point":
        return A["shape_pt"][sl], A["pv_pt"][sl]
    qL, qP = A["q"][mode][0][A["day_id"][t]], A["q"][mode][1][A["day_id"][t]]
    return A["shape_q95"][sl] + qL, np.clip(A["pv_q05"][sl] - qP, 0, None)


def episodes_for(S, A, days=None):
    eps = []
    T = len(A["grid"])
    grp = S.groupby("day").indices
    for day, idx in sorted(grp.items()):
        if days is not None and day not in days:
            continue
        t0 = int(np.searchsorted(A["grid"], day + pd.Timedelta(hours=CFG.EPISODE_START_HOUR)))
        t_end = min(T, t0 + int(CFG.EPISODE_MAX_H * 60 / CFG.STEP_MIN), int(S.d.values[idx].max()))
        if t_end > t0:
            eps.append((day, t0, t_end, np.asarray(idx)))
    return eps


def simulate(method, S, A, feeder, episodes, mode=None, scale_e=1.0, record_ctx=None):
    mode = mode or METHOD_MODE.get(method, CFG.MAIN_CONFORMAL_MODE)
    T = len(A["grid"])
    ev = np.zeros(T)
    delivered = np.zeros(len(S))
    a_all, d_all = S.a.values, np.minimum(S.d.values, T)
    e_all, rmax_all = S.e.values * scale_e, S.rmax.values
    ids_all = S.session_id.values
    stats_ = {"steps": 0, "iters": 0, "uplink": 0, "downlink": 0, "lp_fail": 0, "runtime_s": 0.0, "binding_steps": 0}
    _ADMM_WARM.clear()
    t_start = time.time()
    for day, t0, t_end, idx in episodes:
        a, d, rmax = a_all[idx], np.minimum(d_all[idx], t_end), rmax_all[idx]
        rem = np.minimum(e_all[idx], rmax * np.maximum(d - a, 0) * CFG.DT_H)
        if method == "offline_opt":
            H = t_end - t0
            sl = slice(t0, t_end)
            cap = feeder.ev_cap_kw(A["shape_act"][sl], A["pv_act"][sl], ev[sl])
            X, ok, _ = build_lp(rem, a - t0, d - t0, rmax, cap, A["price"][sl], A["pv_act"][sl])
            ev[sl] += X.sum(0)
            delivered[idx] = X.sum(1) * CFG.DT_H
            continue
        for t in range(max(t0, int(a.min())), t_end):
            pres = np.where((a <= t) & (d > t) & (rem > 1e-6))[0]
            if len(pres) == 0:
                continue
            H = int(min(CFG.HMAX_STEPS, (d[pres] - t).max(), T - t))
            sh_b, pv_b = bounds_for(A, mode, t, H)
            cap = feeder.ev_cap_kw(sh_b, pv_b, ev[t:t + H])
            ctx = {"e": rem[pres], "d": np.minimum(d[pres] - t, H), "rmax": rmax[pres], "cap": cap,
                   "price": A["price"][t:t + H], "pv": pv_b, "ids": ids_all[idx][pres]}
            if record_ctx is not None and np.random.random() < record_ctx["p"] and len(record_ctx["store"]) < record_ctx["n"]:
                record_ctx["store"].append(copy.deepcopy({k_: v for k_, v in ctx.items() if k_ != "ids"}))
            r, info = SCHEDULERS[method](ctx)
            r = project_step(r, ctx["rmax"], ctx["e"], None if method == "uncontrolled" else cap[0])
            if method != "uncontrolled" and r.sum() >= cap[0] - 1e-6 and cap[0] < np.minimum(ctx["rmax"], ctx["e"] / CFG.DT_H).sum():
                stats_["binding_steps"] += 1
            rem[pres] -= r * CFG.DT_H
            ev[t] += r.sum()
            stats_["steps"] += 1
            stats_["iters"] += info.get("iters", 0)
            stats_["uplink"] += info.get("uplink_floats", 0)
            stats_["downlink"] += info.get("downlink_floats", 0)
            stats_["lp_fail"] += info.get("lp_fail", 0)
        delivered[idx] = np.minimum(e_all[idx], rmax * np.maximum(d - a, 0) * CFG.DT_H) - rem
    stats_["runtime_s"] = time.time() - t_start
    return ev, delivered, stats_

In [ ]:
# ---------------------------------------------------------------- evaluation (actual data, AC power flow)
_AC_CACHE = {}


def evaluate(method, ev, delivered, S, A, feeder, episodes, tag=""):
    """Per-day metrics. Voltage/branch violations are confirmed by AC power flow."""
    net_kw = ev - A["pv_act"]
    imp = np.maximum(net_kw, 0)
    vmin_lin, vbase_lin, lpct_lin = feeder.lin_eval(A["shape_act"], np.maximum(net_kw, 0))
    vmin_ac = vmin_lin.copy()
    lpct_ac = lpct_lin.copy()
    cand = np.where((ev > 0.1) & ((vmin_lin < CFG.VMIN + CFG.SCREEN_MARGIN) | (lpct_lin > 90)))[0]
    for t in cand:
        key = (id(feeder), round(float(A["shape_act"][t]), 5), round(float(net_kw[t]), 1))
        if key not in _AC_CACHE:
            _AC_CACHE[key] = feeder.ac_eval(float(A["shape_act"][t]), float(max(net_kw[t], 0)))
        vmin_ac[t], lpct_ac[t] = _AC_CACHE[key]
    v_viol = (ev > 0.1) & (net_kw > 0.1) & (vmin_ac < CFG.VMIN - 1e-4)
    l_viol = (ev > 0.1) & (net_kw > 0.1) & (lpct_ac > 100 + 1e-6)
    t_viol = net_kw > feeder.tr_kw + 0.1
    any_v = v_viol | l_viol | t_viol
    rows = []
    day_ids = A["day_id"].values
    for day, t0, t_end, idx in episodes:
        m = day_ids == day
        req = S.e.values[idx].sum()
        req_feas = np.minimum(S.e.values[idx], S.rmax.values[idx] * np.maximum(np.minimum(S.d.values[idx], t_end) - S.a.values[idx], 0) * CFG.DT_H)
        dl = delivered[idx]
        e_ev = ev[m].sum() * CFG.DT_H
        cost = float(np.sum(A["price"][m] / 1000 * CFG.DT_H * imp[m]))
        rows.append({"method": method, "tag": tag, "day": day, "n_sessions": len(idx),
                     "energy_requested_kwh": req, "energy_delivered_kwh": dl.sum(),
                     "energy_served_pct": 100 * dl.sum() / max(req, 1e-9),
                     "sessions_fully_served_pct": 100 * np.mean(dl >= req_feas - 0.05),
                     "cost_usd": cost, "cost_per_kwh_delivered": cost / max(dl.sum(), 1e-9),
                     "pv_self_consumption_pct": 100 * np.sum(np.minimum(ev[m], A["pv_act"][m])) * CFG.DT_H / max(e_ev, 1e-9),
                     "peak_net_kw": float(net_kw[m].max()), "min_voltage_pu": float(vmin_ac[m][ev[m] > 0.1].min()) if (ev[m] > 0.1).any() else np.nan,
                     "voltage_viol_steps": int(v_viol[m].sum()), "branch_viol_steps": int(l_viol[m].sum()),
                     "transformer_viol_steps": int(t_viol[m].sum()), "any_viol_steps": int(any_v[m].sum()),
                     "base_only_voltage_below_vmin_steps": int(((vbase_lin < CFG.VMIN) & m).sum())})
    return pd.DataFrame(rows), {"ac_runs": len(cand)}


def summarize(daily):
    agg = daily.groupby(["method", "tag"]).agg(
        days=("day", "nunique"), sessions=("n_sessions", "sum"),
        energy_requested_kwh=("energy_requested_kwh", "sum"), energy_delivered_kwh=("energy_delivered_kwh", "sum"),
        cost_usd=("cost_usd", "sum"), any_viol_steps=("any_viol_steps", "sum"),
        voltage_viol_steps=("voltage_viol_steps", "sum"), branch_viol_steps=("branch_viol_steps", "sum"),
        transformer_viol_steps=("transformer_viol_steps", "sum"), min_voltage_pu=("min_voltage_pu", "min"),
        peak_net_kw=("peak_net_kw", "max"), sessions_fully_served_pct=("sessions_fully_served_pct", "mean"),
        pv_self_consumption_pct=("pv_self_consumption_pct", "mean")).reset_index()
    agg["energy_served_pct"] = 100 * agg.energy_delivered_kwh / agg.energy_requested_kwh
    agg["cost_per_kwh_delivered"] = agg.cost_usd / agg.energy_delivered_kwh
    agg["viol_hours"] = agg.any_viol_steps * CFG.DT_H
    return agg

## Run E1 + E6 (data and forecasting) for each site

In [ ]:
def pick_days(S, n_per_month=2, months=range(1, 13)):
    days = pd.Series(sorted(S.day.unique()))
    out = []
    for mth in months:
        wd = days[(days.dt.month == mth) & (days.dt.dayofweek == 2)]       # Wednesdays
        out += list(wd.iloc[[0, 2]] if len(wd) >= 3 and n_per_month == 2 else wd.iloc[:n_per_month])
    return set(out)


SITE = {}
E1_rows, E6_rows, LEAK_rows = [], [], []
for site in CFG.SITES:
    F = build_site_frame(site)
    F, fc, fmet, leak = run_forecasting(site, F)
    A = site_arrays(site, F, fc)
    S, s_stats, conc = prepare_sessions(site, A["grid"])
    if CFG.SMOKE_TEST:
        eval_days = pick_days(S, 1, months=[1, 4, 7, 10])
    elif CFG.QUICK:
        eval_days = pick_days(S, 2)
    else:
        eval_days = None
    SITE[site] = dict(F=F, fc=fc, A=A, S=S, conc=conc, eval_days=eval_days)
    E1_rows.append(s_stats)
    fmet.insert(0, "site", site); leak.insert(0, "site", site)
    E6_rows.append(fmet); LEAK_rows.append(leak)

E1 = pd.DataFrame(E1_rows)
E6 = pd.concat(E6_rows, ignore_index=True)
LEAK = pd.concat(LEAK_rows, ignore_index=True)
save_table(E1, "E1_dataset_statistics"); save_table(E6, "E6_forecast_metrics"); save_table(LEAK, "E6_leakage_audit")
display(E1.T) if "display" in dir() else print(E1.T)
print(E6.round(3).to_string())
print(LEAK.to_string())

In [ ]:
# ---------------------------------------------------------------- E5 coverage of one-sided bounds, by month
COV_rows = []
for site, D in SITE.items():
    A = D["A"]
    month = A["grid"].month.values
    for mode in CFG.CONFORMAL_MODES:
        qL = np.array([A["q"][mode][0][d] for d in A["day_id"].values])
        qP = np.array([A["q"][mode][1][d] for d in A["day_id"].values])
        upL = A["shape_pt"] if mode == "point" else A["shape_q95"] + qL
        loP = A["pv_pt"] if mode == "point" else np.clip(A["pv_q05"] - qP, 0, None)
        okL = A["shape_act"] <= upL
        okP = np.where(A["day_mask"], A["pv_act"] >= loP, np.nan)
        for mth in range(1, 13):
            mm = month == mth
            COV_rows.append({"site": site, "mode": mode, "month": mth,
                             "load_upper_coverage_pct": 100 * okL[mm].mean(),
                             "pv_lower_coverage_pct": 100 * np.nanmean(okP[mm]),
                             "load_margin_mean": float(np.mean(upL[mm] - A["shape_pt"][mm])),
                             "pv_margin_mean_kw": float(np.nanmean(np.where(A["day_mask"][mm], A["pv_pt"][mm] - loP[mm], np.nan)))})
COV = pd.DataFrame(COV_rows)
save_table(COV, "E5_monthly_coverage_by_mode")
print(COV.groupby(["site", "mode"])[["load_upper_coverage_pct", "pv_lower_coverage_pct"]].mean().round(2))

## Run E2: build the feeder and validate LinDistFlow against AC power flow

In [ ]:
E2_rows = []
for site, D in SITE.items():
    A = D["A"]
    feeder = Feeder(A["shape_train"], CFG.V_BASE_TARGET, CFG.SITE_INFO[site]["transformer_kw"])
    D["feeder"] = feeder
    rng = np.random.default_rng(CFG.SEED)
    ts = rng.choice(len(A["grid"]), size=CFG.AC_VALIDATION_SAMPLES, replace=False)
    gk = rng.uniform(0, feeder.tr_kw, size=len(ts))
    lin = feeder.lin_eval(A["shape_act"][ts], gk)[0]
    ac = np.array([feeder.ac_eval(float(A["shape_act"][t]), float(g))[0] for t, g in zip(ts, gk)])
    err = lin - ac
    vb = feeder.lin_eval(A["shape_act"], np.zeros(len(A["grid"])))[1]
    E2_rows.append({"site": site, "base_load_scale": feeder.scale, "loss_margin_v2": feeder.margin,
                    "linDistFlow_minus_AC_vmin_mean": err.mean(), "linDistFlow_minus_AC_vmin_p95": np.quantile(err, 0.95),
                    "linDistFlow_minus_AC_vmin_max": err.max(), "screen_margin": CFG.SCREEN_MARGIN,
                    "screen_is_safe": bool(err.max() < CFG.SCREEN_MARGIN),
                    "test_base_min_voltage": vb.min(), "test_steps_base_below_vmin": int((vb < CFG.VMIN).sum()),
                    "garage_bus": CFG.GARAGE_BUS, "transformer_kw": feeder.tr_kw,
                    "voltage_sensitivity_pu_per_100kW_at_garage": float(feeder.s[CFG.GARAGE_BUS] * 0.1 / 2)})
E2 = pd.DataFrame(E2_rows)
save_table(E2, "E2_network_validation")
print(E2.T)

## Run E3 + E4: main comparison (identical information and constraints)

In [ ]:
MAIN_DAILY, MAIN_STATS, CONTEXTS = [], [], {}
for site, D in SITE.items():
    S, A, feeder = D["S"], D["A"], D["feeder"]
    eps = episodes_for(S, A, D["eval_days"])
    D["episodes"] = eps
    CONTEXTS[site] = {"p": 0.05, "n": CFG.N_CONTEXTS_CONVERGENCE, "store": []}
    D["ev"] = {}
    for method in CFG.MAIN_METHODS:
        ev, dl, st = simulate(method, S, A, feeder, eps,
                              record_ctx=CONTEXTS[site] if method == CFG.PROPOSED else None)
        daily, ac = evaluate(method, ev, dl, S, A, feeder, eps, tag="main")
        daily.insert(0, "site", site)
        MAIN_DAILY.append(daily)
        D["ev"][method] = ev
        st.update({"site": site, "method": method, "ac_runs": ac["ac_runs"]})
        MAIN_STATS.append(st)
        log.info(f"[{site}] {method:15s} {st['runtime_s']:7.1f}s steps={st['steps']} binding={st['binding_steps']}")
MAIN_DAILY = pd.concat(MAIN_DAILY, ignore_index=True)
MAIN = summarize(MAIN_DAILY.drop(columns="site").assign(method=MAIN_DAILY.site + "|" + MAIN_DAILY.method))
MAIN[["site", "method"]] = MAIN.method.str.split("|", expand=True)
RUNSTATS = pd.DataFrame(MAIN_STATS)
RUNSTATS["ms_per_decision"] = 1000 * RUNSTATS.runtime_s / RUNSTATS.steps.clip(lower=1)
RUNSTATS["admm_iters_per_decision"] = RUNSTATS.iters / RUNSTATS.steps.clip(lower=1)
RUNSTATS["binding_share_pct"] = 100 * RUNSTATS.binding_steps / RUNSTATS.steps.clip(lower=1)
save_table(MAIN_DAILY, "E4_main_daily"); save_table(MAIN, "E4_main_summary"); save_table(RUNSTATS, "E4_runtime_and_protocol")
cols = ["site", "method", "days", "energy_served_pct", "sessions_fully_served_pct", "cost_per_kwh_delivered",
        "pv_self_consumption_pct", "viol_hours", "voltage_viol_steps", "branch_viol_steps", "transformer_viol_steps",
        "min_voltage_pu", "peak_net_kw"]
print(MAIN[cols].round(4).to_string())
print(RUNSTATS[["site", "method", "steps", "binding_share_pct", "ms_per_decision", "admm_iters_per_decision", "lp_fail"]].round(2).to_string())

## Run E5: closed-loop ablation (proposed protocol × each conformal mode)
Same protocol and the same days; only the uncertainty margin changes. If online recalibration matters, it shows up here as fewer AC violations, or as less cost and more energy for the same safety.

In [ ]:
E5_DAILY = []
for site, D in SITE.items():
    S, A, feeder, eps = D["S"], D["A"], D["feeder"], D["episodes"]
    for mode in CFG.CONFORMAL_MODES:
        if mode == CFG.MAIN_CONFORMAL_MODE:
            daily = MAIN_DAILY[(MAIN_DAILY.site == site) & (MAIN_DAILY.method == CFG.PROPOSED)].copy()
            daily["tag"] = mode
        else:
            ev, dl, st = simulate(CFG.PROPOSED, S, A, feeder, eps, mode=mode)
            daily, _ = evaluate(CFG.PROPOSED, ev, dl, S, A, feeder, eps, tag=mode)
            daily.insert(0, "site", site)
        E5_DAILY.append(daily)
        log.info(f"[{site}] E5 mode={mode} done")
E5_DAILY = pd.concat(E5_DAILY, ignore_index=True)
E5 = summarize(E5_DAILY.drop(columns="site").assign(tag=E5_DAILY.site + "|" + E5_DAILY.tag))
E5[["site", "mode"]] = E5.tag.str.split("|", expand=True)
E5_DAILY["month"] = pd.to_datetime(E5_DAILY.day).dt.month
E5_MONTH = E5_DAILY.groupby(["site", "tag", "month"]).agg(viol_steps=("any_viol_steps", "sum"),
                                                         cost=("cost_usd", "sum"), energy=("energy_delivered_kwh", "sum")).reset_index()
save_table(E5_DAILY, "E5_ablation_daily"); save_table(E5, "E5_ablation_summary"); save_table(E5_MONTH, "E5_ablation_monthly")
print(E5[["site", "mode", "energy_served_pct", "cost_per_kwh_delivered", "viol_hours", "min_voltage_pu"]].round(4).to_string())

## E3 analysis: ADMM convergence, optimality gap vs the centralized LP, scalability, communication

In [ ]:
CONV_rows, TRACE = [], []
for site in SITE:
    for j, ctx in enumerate(CONTEXTS[site]["store"]):
        N = len(ctx["e"])
        t1 = time.time(); X_lp, ok, _ = build_lp(ctx["e"], np.zeros(N, int), ctx["d"], ctx["rmax"], ctx["cap"], ctx["price"], ctx["pv"]); t_lp = time.time() - t1
        f_lp = lp_objective(X_lp, ctx["e"], ctx["cap"], ctx["price"], ctx["pv"])
        e_sched = max(X_lp.sum() * CFG.DT_H, 1e-6)
        value = max(np.mean(np.maximum(ctx["price"], 0)) / 1000 * ctx["e"].sum(), 1e-6)
        row = {"site": site, "ctx": j, "n_evs": N, "horizon": len(ctx["cap"]), "lp_ok": ok, "obj_lp": f_lp,
               "t_lp_ms": None}
        row["t_lp_ms"] = 1000 * t_lp
        for tag, eps_ in [("", CFG.ADMM_EPS), ("_tight", CFG.ADMM_EPS_TIGHT)]:
            t1 = time.time(); X_ad, info = admm_solve(ctx["e"], ctx["d"], ctx["rmax"], ctx["cap"], ctx["price"], ctx["pv"], eps=eps_, trace=(tag == "")); t_ad = time.time() - t1
            over = X_ad.sum(0) > ctx["cap"] + 1e-9
            X_ad[:, over] *= (ctx["cap"][over] / X_ad.sum(0)[over])[None, :]
            f_ad = lp_objective(X_ad, ctx["e"], ctx["cap"], ctx["price"], ctx["pv"])
            row.update({f"obj_admm{tag}": f_ad, f"gap_usd_per_kwh{tag}": (f_ad - f_lp) / e_sched,
                        f"gap_pct_of_energy_value{tag}": 100 * (f_ad - f_lp) / value,
                        f"iters{tag}": info["iters"], f"t_admm_ms{tag}": 1000 * t_ad,
                        f"uplink_floats_per_ev{tag}": info["uplink_floats"] / max(N, 1)})
            if tag == "" and j < 12:
                TRACE += [{"site": site, "ctx": j, "iter": i, "r_pri": a_, "r_dual": b_} for i, a_, b_ in info["hist"]]
        CONV_rows.append(row)
CONV = pd.DataFrame(CONV_rows); TRACE = pd.DataFrame(TRACE)
save_table(CONV, "E3_admm_vs_lp_gap"); save_table(TRACE, "E3_admm_residual_traces")
print(CONV.describe().T[["mean", "50%", "max"]].round(5))

# scalability on synthetic decision contexts (runtime only; clearly labelled)
SCAL_rows = []
rng = np.random.default_rng(CFG.SEED)
H = 48
cap_ref = feeder.ev_cap_kw(np.full(H, 0.85), np.zeros(H))
for N in [10, 50, 100, 250, 500, 1000]:
    e = rng.uniform(2, 30, N); d = rng.integers(8, H + 1, N); rmax = np.full(N, CFG.EVSE_MAX_KW)
    cap = cap_ref * N / 50; price = 30 + 10 * np.sin(np.arange(H) / 8); pv = np.zeros(H)
    t1 = time.time(); X_lp, ok, _ = build_lp(e, np.zeros(N, int), d, rmax, cap, price, pv); t_lp = time.time() - t1
    t1 = time.time(); X_ad, info = admm_solve(e, d, rmax, cap, price, pv); t_ad = time.time() - t1
    SCAL_rows.append({"n_evs": N, "horizon": H, "t_central_lp_s": t_lp, "t_admm_total_s": t_ad,
                      "t_admm_per_agent_ms": 1000 * t_ad / N, "iters": info["iters"],
                      "gap_usd_per_kwh": (lp_objective(X_ad, e, cap, price, pv) - lp_objective(X_lp, e, cap, price, pv)) / max(X_lp.sum() * CFG.DT_H, 1e-6),
                      "note": "synthetic decision contexts, runtime study only"})
SCAL = pd.DataFrame(SCAL_rows)
save_table(SCAL, "E3_scalability_synthetic_contexts")
print(SCAL.round(4).to_string())

## E2 stress tests: when do the constraints bind and who violates them?
Sweeps feeder headroom (`V_BASE_TARGET`), transformer rating, and EV demand. Penetration > 1 scales each real session's energy; arrivals are **not** synthesized. Runs on `STRESS_N_DAYS` weekdays spread over the year.

In [ ]:
STRESS_rows = []
for site, D in SITE.items():
    S, A = D["S"], D["A"]
    days_all = sorted(S.day.unique())
    wk = [d for d in days_all if pd.Timestamp(d).dayofweek < 5]
    sel = set(wk[i] for i in np.linspace(0, len(wk) - 1, CFG.STRESS_N_DAYS).astype(int))
    eps = episodes_for(S, A, sel)
    for vt in CFG.STRESS_V_TARGETS:
        for trk in CFG.STRESS_TR_KW:
            fdr = Feeder(A["shape_train"], vt, trk)
            for pen in CFG.STRESS_PENETRATION:
                for method in CFG.STRESS_METHODS:
                    ev, dl, st = simulate(method, S, A, fdr, eps, scale_e=pen)
                    Sx = S.assign(e=S.e * pen)
                    daily, _ = evaluate(method, ev, dl, Sx, A, fdr, eps, tag=f"{vt}|{trk}|{pen}")
                    sm = summarize(daily).iloc[0].to_dict()
                    sm.update({"site": site, "v_base_target": vt, "transformer_kw": trk, "penetration": pen,
                               "binding_share_pct": 100 * st["binding_steps"] / max(st["steps"], 1)})
                    STRESS_rows.append(sm)
        log.info(f"[{site}] stress V_target={vt} done")
STRESS = pd.DataFrame(STRESS_rows)
save_table(STRESS, "E2_stress_tests")
print(STRESS[["site", "method", "v_base_target", "transformer_kw", "penetration", "energy_served_pct",
              "cost_per_kwh_delivered", "viol_hours", "binding_share_pct"]].round(3).to_string())

## E7: Paired statistics over days (proposed vs each baseline, per site)
* Unit of analysis = **operating day** (paired: same sessions, same weather, same prices).
* 95% CI of the mean difference from a **moving-block bootstrap** (7-day blocks, which keeps weekly autocorrelation).
* Wilcoxon signed-rank p-values with **Holm** correction across all comparisons for each metric.
* Caltech and JPL share CAISO load and price and have nearby NSRDB cells. They are presented as **two demand patterns on the same feeder model**, not as two climates.

In [ ]:
def block_bootstrap_ci(x, n_boot=CFG.N_BOOT, block=CFG.BLOCK_DAYS, seed=CFG.SEED):
    x = np.asarray(x, float)
    n = len(x)
    if n < 3:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    nb = int(np.ceil(n / block))
    starts = rng.integers(0, max(n - block + 1, 1), size=(n_boot, nb))
    idx = (starts[:, :, None] + np.arange(block)[None, None, :]).reshape(n_boot, -1)[:, :n] % n
    means = x[idx].mean(1)
    return float(np.quantile(means, 0.025)), float(np.quantile(means, 0.975))


def holm(p):
    p = np.asarray(p, float)
    order = np.argsort(p)
    adj = np.empty_like(p)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, min(1.0, (len(p) - rank) * p[i]))
        adj[i] = running
    return adj


METRICS = [("cost_per_kwh_delivered", "lower"), ("energy_served_pct", "higher"),
           ("any_viol_steps", "lower"), ("pv_self_consumption_pct", "higher"), ("peak_net_kw", "lower")]
ST_rows = []
for site in SITE:
    dd = MAIN_DAILY[MAIN_DAILY.site == site]
    piv = {m: dd[dd.method == m].set_index("day") for m in CFG.MAIN_METHODS}
    for base in [m for m in CFG.MAIN_METHODS if m != CFG.PROPOSED]:
        for met, better in METRICS:
            a = piv[CFG.PROPOSED][met]; b = piv[base][met]
            j = a.index.intersection(b.index)
            diff = (a.loc[j] - b.loc[j]).dropna().sort_index()
            lo, hi = block_bootstrap_ci(diff.values)
            try:
                p = stats.wilcoxon(diff.values, zero_method="zsplit").pvalue if np.any(diff.values != 0) else 1.0
            except ValueError:
                p = np.nan
            ST_rows.append({"site": site, "proposed": CFG.PROPOSED, "baseline": base, "metric": met, "better_if": better,
                            "n_days": len(diff), "mean_proposed": a.loc[j].mean(), "mean_baseline": b.loc[j].mean(),
                            "mean_diff": diff.mean(), "ci95_lo": lo, "ci95_hi": hi, "wilcoxon_p": p})
STATS = pd.DataFrame(ST_rows)
STATS["p_holm"] = np.nan
for met in STATS.metric.unique():
    m = (STATS.metric == met) & STATS.wilcoxon_p.notna()
    STATS.loc[m, "p_holm"] = holm(STATS.loc[m, "wilcoxon_p"].values)


def verdict(r):
    if np.isnan(r.ci95_lo):
        return "n/a"
    if r.ci95_lo <= 0 <= r.ci95_hi or (r.p_holm > 0.05 if pd.notna(r.p_holm) else True):
        return "no significant difference"
    good = (r.mean_diff < 0) if r.better_if == "lower" else (r.mean_diff > 0)
    return "proposed better" if good else "proposed worse"


STATS["verdict"] = STATS.apply(verdict, axis=1)
save_table(STATS, "E7_paired_statistics")
print(STATS[["site", "baseline", "metric", "n_days", "mean_diff", "ci95_lo", "ci95_hi", "p_holm", "verdict"]].round(4).to_string())

## Figures (300 dpi PNG, one folder per site)

In [ ]:
for site, D in SITE.items():
    A, fc = D["A"], D["fc"]
    # Fig 1: forecasts with CQR bands (3 test days, day-ahead)
    fig, ax = plt.subplots(2, 1, figsize=(10, 6))
    te = (D["F"].index >= CFG.TEST_START)
    sel = np.where(te)[0][96 * 150: 96 * 153]
    for k_, (tgt, col, unit) in enumerate([("pv", "ghi", "GHI (W/m²)"), ("load", "load_mw", "Load (MW)")]):
        o = fc[(tgt, "day_ahead")]
        ix = D["F"].index[sel]
        ax[k_].fill_between(ix, o["lo_c"][sel], o["hi_c"][sel], alpha=0.25, color="#2a9d8f", label="95% CQR")
        ax[k_].plot(ix, D["F"][col].values[sel], color="k", lw=1, label="actual")
        ax[k_].plot(ix, o["P"]["point"].values[sel], color="#e76f51", lw=1, label="day-ahead forecast")
        ax[k_].set_ylabel(unit); ax[k_].legend(fontsize=7, loc="upper right")
    fig.suptitle(f"{site}: day-ahead forecasts with conformalized quantile intervals")
    save_fig(fig, "Fig01_forecasts", site)
    # Fig 2: monthly coverage of one-sided load bound by mode
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    cv = COV[COV.site == site]
    for mode in CFG.CONFORMAL_MODES:
        c = cv[cv["mode"] == mode]
        ax[0].plot(c.month, c.load_upper_coverage_pct, marker="o", ms=3, label=mode)
        ax[1].plot(c.month, c.pv_lower_coverage_pct, marker="o", ms=3, label=mode)
    for a_ in ax:
        a_.axhline(100 * (1 - CFG.ALPHA), ls="--", c="k", lw=0.8); a_.set_xlabel("month 2019")
    ax[0].set_ylabel("coverage of load upper bound (%)"); ax[1].set_ylabel("coverage of PV lower bound (%)")
    ax[1].legend(fontsize=7)
    fig.suptitle(f"{site}: calibration under seasonal shift")
    save_fig(fig, "Fig02_monthly_coverage", site)
    # Fig 3: one busy day, EV load vs cap for the main methods
    ms = MAIN_DAILY[(MAIN_DAILY.site == site) & (MAIN_DAILY.method == "uncontrolled")].sort_values("energy_requested_kwh")
    if len(ms):
        day = ms.day.iloc[-1]
        m = np.asarray(A["day_id"] == day)
        ix = A["grid"][m]
        cap_true = D["feeder"].ev_cap_kw(A["shape_act"][m], A["pv_act"][m])
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.fill_between(ix, 0, cap_true, color="#dddddd", label="true network limit (hindsight)")
        for meth in ["uncontrolled", "mpc_point", "mpc_conformal", "admm_conformal", "offline_opt"]:
            ax.plot(ix, D["ev"][meth][m], lw=1.3, color=PALETTE[meth], label=LABEL[meth])
        ax.plot(ix, A["pv_act"][m], ls=":", c="#f2c14e", label="PV")
        ax.set_ylabel("kW"); ax.legend(fontsize=7, ncol=2)
        fig.suptitle(f"{site}: busiest evaluated day ({pd.Timestamp(day).date()})")
        save_fig(fig, "Fig03_busiest_day_profiles", site)
    # Fig 4: cost vs violations (main methods)
    mm = MAIN[MAIN.site == site]
    fig, ax = plt.subplots(figsize=(6, 4.5))
    for _, r in mm.iterrows():
        ax.scatter(r.viol_hours, r.cost_per_kwh_delivered, s=60, color=PALETTE[r.method], label=LABEL[r.method])
    ax.set_xlabel("constraint-violation hours (AC power flow)"); ax.set_ylabel("cost per kWh delivered ($)")
    ax.legend(fontsize=7)
    fig.suptitle(f"{site}: cost vs network safety")
    save_fig(fig, "Fig04_cost_vs_violations", site)
    # Fig 5: E5 ablation bars
    e5 = E5[E5.site == site].set_index("mode").reindex(CFG.CONFORMAL_MODES)
    fig, ax = plt.subplots(1, 3, figsize=(12, 3.6))
    for a_, col, lab in zip(ax, ["viol_hours", "cost_per_kwh_delivered", "energy_served_pct"],
                            ["violation hours", "$ per kWh delivered", "energy served (%)"]):
        a_.bar(e5.index, e5[col], color="#264653"); a_.set_title(lab, fontsize=9); a_.tick_params(axis="x", rotation=35)
    fig.suptitle(f"{site}: closed-loop ablation (proposed protocol)")
    save_fig(fig, "Fig05_E5_ablation", site)
    # Fig 6: voltage distribution
    fig, ax = plt.subplots(figsize=(7, 4))
    dm = MAIN_DAILY[MAIN_DAILY.site == site]
    data = [dm[dm.method == m].min_voltage_pu.dropna().values for m in CFG.MAIN_METHODS]
    ax.boxplot(data, tick_labels=[LABEL[m].replace(" ", "\n", 1) for m in CFG.MAIN_METHODS], showfliers=False)
    ax.axhline(CFG.VMIN, c="r", ls="--", lw=0.8); ax.set_ylabel("daily minimum feeder voltage (p.u., AC)")
    ax.tick_params(axis="x", labelsize=6)
    fig.suptitle(f"{site}: daily minimum voltage while EVs charge")
    save_fig(fig, "Fig06_voltage_distribution", site)

# combined: ADMM convergence + scalability + stress
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
for (s_, c_), g in TRACE.groupby(["site", "ctx"]):
    ax[0].semilogy(g["iter"], g.r_pri.clip(lower=1e-4), lw=0.7, alpha=0.6)
ax[0].set_xlabel("ADMM iteration"); ax[0].set_ylabel("primal residual (normalized)")
ax[1].loglog(SCAL.n_evs, SCAL.t_central_lp_s, "o-", label="centralized LP (total)")
ax[1].loglog(SCAL.n_evs, SCAL.t_admm_per_agent_ms / 1000, "s-", label="ADMM per agent")
ax[1].set_xlabel("EVs in decision"); ax[1].set_ylabel("seconds"); ax[1].legend(fontsize=7)
if len(CONV):
    ax[2].hist(CONV.gap_pct_of_energy_value, bins=30, color="#264653", alpha=0.8, label=f"eps={CFG.ADMM_EPS}")
    ax[2].hist(CONV.gap_pct_of_energy_value_tight, bins=30, color="#e76f51", alpha=0.6, label=f"eps={CFG.ADMM_EPS_TIGHT}")
    ax[2].set_xlabel("ADMM gap vs centralized LP (% of energy value)"); ax[2].legend(fontsize=7)
fig.suptitle("Decentralized protocol: convergence, scalability, optimality gap")
save_fig(fig, "Fig07_admm_protocol")
if len(STRESS):
    for site in SITE:
        sub = STRESS[(STRESS.site == site) & (STRESS.penetration == max(CFG.STRESS_PENETRATION))]
        fig, ax = plt.subplots(1, len(CFG.STRESS_TR_KW), figsize=(4 * len(CFG.STRESS_TR_KW), 3.6), sharey=True)
        ax = np.atleast_1d(ax)
        for a_, trk in zip(ax, CFG.STRESS_TR_KW):
            for m in CFG.STRESS_METHODS:
                g = sub[(sub.transformer_kw == trk) & (sub.method == m)].sort_values("v_base_target", ascending=False)
                a_.plot(g.v_base_target, g.viol_hours, marker="o", color=PALETTE[m], label=LABEL[m])
            a_.invert_xaxis(); a_.set_title(f"transformer {trk:.0f} kW", fontsize=9); a_.set_xlabel("base-case worst voltage (p.u.)")
        ax[0].set_ylabel("violation hours"); ax[-1].legend(fontsize=6)
        fig.suptitle(f"{site}: stress test at {max(CFG.STRESS_PENETRATION)}x demand")
        save_fig(fig, "Fig08_stress", site)
print("figures written to", CFG.OUT.resolve())

## Week 9: red-team checks
Each check is something a hostile reviewer would try. A **FAIL** must be fixed or disclosed before submission.

In [ ]:
RT = []
def check(name, ok, detail):
    RT.append({"check": name, "result": "PASS" if ok else "FAIL", "detail": detail})

check("Data mode is real", not CFG.SMOKE_TEST, DATA_MODE)
check("No forecast feature closer than horizon (leakage)", bool(LEAK.ok.all()),
      LEAK[["site", "target", "horizon", "min_feature_lag_steps", "horizon_steps"]].to_dict("records"))
check("Train < calibration < test (chronological)", pd.Timestamp(CFG.TRAIN_END) < pd.Timestamp(CFG.CAL_START) < pd.Timestamp(CFG.TEST_START),
      f"{CFG.TRAIN_START}..{CFG.TRAIN_END} | {CFG.CAL_START}..{CFG.CAL_END} | {CFG.TEST_START}..{CFG.TEST_END}")
check("Load normalization uses training data only", True, "norm = max(load) over the training period (site_arrays)")
for site in SITE:
    s1 = E1[E1.site == site].iloc[0]
    check(f"[{site}] sessions are real and on real dates", not CFG.SMOKE_TEST,
          f"{s1.sessions_in_test_year} sessions, {s1.sessions_per_weekday_mean}/weekday, peak {s1.peak_concurrent_EVs} concurrent")
    rs = RUNSTATS[(RUNSTATS.site == site) & (RUNSTATS.method == CFG.PROPOSED)].iloc[0]
    check(f"[{site}] network limit actually binds for the proposed method", rs.binding_share_pct > 1.0,
          f"binding in {rs.binding_share_pct:.1f}% of decision steps (if ~0, use a stress-test setting as the main case)")
    e2 = E2[E2.site == site].iloc[0]
    check(f"[{site}] AC screening margin larger than LinDistFlow error", bool(e2.screen_is_safe),
          f"max error {e2.linDistFlow_minus_AC_vmin_max:.4f} p.u. vs margin {CFG.SCREEN_MARGIN}")
    check(f"[{site}] LP never failed", int(RUNSTATS[RUNSTATS.site == site].lp_fail.sum()) == 0,
          f"{int(RUNSTATS[RUNSTATS.site == site].lp_fail.sum())} fallbacks to EDF")
    off = MAIN[(MAIN.site == site) & (MAIN.method == "offline_opt")].iloc[0]
    prop = MAIN[(MAIN.site == site) & (MAIN.method == CFG.PROPOSED)].iloc[0]
    check(f"[{site}] offline optimum is not beaten on cost at equal energy (sanity)",
          not (prop.cost_per_kwh_delivered < off.cost_per_kwh_delivered - 1e-4 and prop.energy_served_pct >= off.energy_served_pct - 0.1),
          f"proposed {prop.cost_per_kwh_delivered:.4f} $/kWh @ {prop.energy_served_pct:.2f}% vs offline {off.cost_per_kwh_delivered:.4f} @ {off.energy_served_pct:.2f}%")
check("Same constraints for all controlled online methods", True,
      "every method calls Feeder.ev_cap_kw with the same bounds; only mpc_point uses point bounds (by design, documented)")
check("Main conformal mode fixed a priori", CFG.MAIN_CONFORMAL_MODE == "rolling", f"MAIN_CONFORMAL_MODE={CFG.MAIN_CONFORMAL_MODE}")
if len(CONV):
    check("ADMM converges to the centralized LP optimum (tight tolerance, p95 gap < 1% of energy value)",
          float(CONV.gap_pct_of_energy_value_tight.quantile(0.95)) < 1.0,
          f"tight eps={CFG.ADMM_EPS_TIGHT}: median {CONV.gap_pct_of_energy_value_tight.median():.4f}%, p95 {CONV.gap_pct_of_energy_value_tight.quantile(0.95):.4f}% | "
          f"simulation eps={CFG.ADMM_EPS}: p95 {CONV.gap_pct_of_energy_value.quantile(0.95):.4f}%")
for site in SITE:
    a_ = MAIN[(MAIN.site == site) & (MAIN.method == CFG.PROPOSED)].iloc[0]
    b_ = MAIN[(MAIN.site == site) & (MAIN.method == "mpc_conformal")].iloc[0]
    rel = 100 * (a_.cost_per_kwh_delivered - b_.cost_per_kwh_delivered) / max(b_.cost_per_kwh_delivered, 1e-9)
    check(f"[{site}] decentralized result matches centralized MPC with the same information (|cost diff| < 5%)", abs(rel) < 5,
          f"proposed {a_.cost_per_kwh_delivered:.4f} vs MPC-conformal {b_.cost_per_kwh_delivered:.4f} $/kWh ({rel:+.2f}%), energy {a_.energy_served_pct:.3f}% vs {b_.energy_served_pct:.3f}%")
check("Statistics use paired days, block bootstrap, Holm", True, f"block={CFG.BLOCK_DAYS} days, n_boot={CFG.N_BOOT}")
check("Sites share weather/grid inputs (must be disclosed)", True, "Caltech & JPL: same CAISO load/price; NSRDB cells ~7 km apart")
REDTEAM = pd.DataFrame(RT)
save_table(REDTEAM, "W9_red_team_checks")
print(REDTEAM.to_string())

## Reviewer-response matrix and results summary (auto-filled)
Text in `[[...]]` is for you to write. Every number comes from the tables above.

In [ ]:
def g(df, **kw):
    m = np.ones(len(df), bool)
    for k_, v in kw.items():
        m &= (df[k_] == v).values
    return df[m].iloc[0] if m.any() else None


lines = [f"# Reviewer-response matrix ({DATA_MODE})", "",
         f"Generated {time.strftime('%Y-%m-%d %H:%M')} | sites: {', '.join(SITE)} | days per site: "
         + ", ".join(f"{s}={MAIN[MAIN.site == s].days.iloc[0]}" for s in SITE), ""]
for site in SITE:
    P = g(MAIN, site=site, method=CFG.PROPOSED); MP = g(MAIN, site=site, method="mpc_point")
    MC = g(MAIN, site=site, method="mpc_conformal"); OF = g(MAIN, site=site, method="offline_opt")
    ED = g(MAIN, site=site, method="edf"); UN = g(MAIN, site=site, method="uncontrolled")
    e1 = g(E1, site=site); e2 = g(E2, site=site); rs = g(RUNSTATS, site=site, method=CFG.PROPOSED)
    pvf = E6[(E6.site == site) & (E6.target == "pv") & (E6.horizon == "day_ahead")]
    pv_gbm = pvf[pvf.model.isin(["LightGBM", "HistGBM"])].iloc[0]
    pv_cqr = pvf[pvf.model == "interval: CQR"].iloc[0]
    cg = CONV[CONV.site == site]
    lines += [f"## {site}", "",
              "| Comment | What we did | Evidence (numbers) |", "|---|---|---|",
              f"| R1.1 / R3.1 novelty | [[contrast with TSG 2025 grid-aware scheduling: decentralized + conformal margins + online recalibration]] | ADMM within {cg.gap_pct_of_energy_value.quantile(0.95):.3f}% (p95, of energy value) of the centralized LP; EV data never leave the agent |",
              f"| R1.1 / R3.2 negotiation protocol | ADMM sharing: EVs upload profiles, coordinator broadcasts congestion price | {rs.admm_iters_per_decision:.1f} iterations / decision, {rs.ms_per_decision:.1f} ms / decision |",
              f"| R1.2 / R2.1 / R3.5 network | IEEE 33-bus, garage at bus {CFG.GARAGE_BUS}, AC power-flow evaluation | LinDistFlow-AC error ≤ {e2.linDistFlow_minus_AC_vmin_max:.4f} p.u.; violation hours: proposed {P.viol_hours:.2f}, MPC-point {MP.viol_hours:.2f}, uncontrolled {UN.viol_hours:.2f} |",
              f"| R1.3 / R2.6 forecasting | horizon stated (1 h, day-ahead), lag ≥ horizon asserted, smart persistence | day-ahead PV nRMSE {pv_gbm.nRMSE_pct:.1f}%, skill {pv_gbm.skill_vs_persistence:.3f}; CQR coverage {pv_cqr.coverage_pct:.1f}% |",
              f"| R1.4 / R2.2 / R3.3 closed loop | conformal modes compared on operational outcomes (E5) | see E5 table: violation hours by mode; [[state which mode helps and by how much]] |",
              f"| R1.5 / R2.4 / R3.4 baselines | EDF, LLF, MPC point & conformal, offline LP; same information and constraints; cost per kWh delivered | $/kWh: proposed {P.cost_per_kwh_delivered:.4f}, MPC-conformal {MC.cost_per_kwh_delivered:.4f}, offline {OF.cost_per_kwh_delivered:.4f}, EDF {ED.cost_per_kwh_delivered:.4f} |",
              f"| R2.3 grid scaling / sites | CAISO shape normalized by training max, feeder-scaled; shared inputs disclosed | base scale {e2.base_load_scale:.3f}; {e1.sessions_in_test_year} real sessions, peak {e1.peak_concurrent_EVs} concurrent |",
              f"| R2.5 statistics | day-paired, block bootstrap, Wilcoxon + Holm | see E7_paired_statistics.csv |",
              f"| R2.7 completeness | [[all sections, limitations, fixed cross-references]] | — |", ""]
lines += ["## Statistical verdicts (proposed vs baseline)", "",
          STATS[["site", "baseline", "metric", "n_days", "mean_diff", "ci95_lo", "ci95_hi", "p_holm", "verdict"]].round(4).to_markdown(index=False), "",
          "## Red-team checks", "", REDTEAM.to_markdown(index=False)]
(CFG.OUT / "reviewer_response_matrix.md").write_text("\n".join(lines))

summary = [f"# Results summary ({DATA_MODE})", "", "## E1 data", E1.to_markdown(index=False), "",
           "## E6 forecasting (test year)", E6.round(4).to_markdown(index=False), "",
           "## E2 network validation", E2.round(5).to_markdown(index=False), "",
           "## E4 main comparison", MAIN[cols].round(4).to_markdown(index=False), "",
           "## E3 protocol", RUNSTATS[["site", "method", "steps", "binding_share_pct", "ms_per_decision", "admm_iters_per_decision"]].round(3).to_markdown(index=False), "",
           "## E5 closed-loop ablation", E5[["site", "mode", "energy_served_pct", "cost_per_kwh_delivered", "viol_hours", "min_voltage_pu"]].round(4).to_markdown(index=False), "",
           "## E2 stress tests", STRESS[["site", "method", "v_base_target", "transformer_kw", "penetration", "energy_served_pct", "cost_per_kwh_delivered", "viol_hours", "binding_share_pct"]].round(3).to_markdown(index=False)]
(CFG.OUT / "results_summary.md").write_text("\n".join(summary))

manifest = {"data_mode": DATA_MODE, "quick": CFG.QUICK, "time": time.strftime("%Y-%m-%d %H:%M:%S"),
            "python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
            "scipy": scipy.__version__, "pandapower": pp.__version__, "lightgbm": HAS_LGB, "pvlib": HAS_PVLIB,
            "config": {k_: str(v) for k_, v in vars(CFG).items() if not k_.startswith("_")},
            "input_files_sha256": {str(p): hashlib.sha256(open(p, "rb").read()).hexdigest()[:16]
                                   for p in sorted(CFG.DATA.rglob("*")) if p.is_file()}}
(CFG.OUT / "run_manifest.json").write_text(json.dumps(manifest, indent=2, default=str))
print(open(CFG.OUT / "reviewer_response_matrix.md").read()[:4000])
print("\nAll outputs in", CFG.OUT.resolve())